# Part 5 · Interface, score, audio, export & quantum report

Cells **P5-00 … P5-12** belong to Person 5. Every top-level name they define starts with
`p5_` / `P5` (including import aliases), so they can be pasted into the team notebook
without clashing with Parts 1-4. They only *call* the team pipeline:

```python
harmonize_pipeline(melody, style=..., style_weight=..., solver=..., progress=...) -> (result, report)
```

If `harmonize_pipeline` is not defined yet, cell **P5-10** supplies a clearly labelled *mock*
so the interface can be tested. Delete that cell when the real pipeline is in the notebook.
Run `p5_selfcheck()` (cell P5-11) after merging.

In [1]:
# P5-00 Setup: imports, configuration, UI strings
import base64 as p5_b64
import datetime as p5_dt
import html as p5_html
import inspect as p5_inspect
import io as p5_io
import math as p5_math
import os as p5_os
import re as p5_re
import tempfile as p5_tmp
import time as p5_time
import traceback as p5_tb
import wave as p5_wave
from dataclasses import dataclass as p5_dataclass, field as p5_field

import numpy as p5_np
import ipywidgets as p5_w
from IPython.display import display as p5_display, HTML as p5_HTML, Image as p5_Image
from matplotlib.figure import Figure as p5_Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg as p5_Canvas
from matplotlib.patches import Ellipse as p5_Ellipse, Rectangle as p5_Rect, Circle as p5_Circle, PathPatch as p5_PathPatch
from matplotlib.path import Path as p5_Path

try:
    import music21 as p5_m21          # needed for MIDI / MusicXML export and MIDI upload
except Exception:
    p5_m21 = None

P5_LANG = "en"                        # "en" or "zh" - change before calling p5_launch()
P5_OUT_DIR = "p5_output"              # exported files are saved here
P5_MAX_NOTES = 128                    # supports the complete O Canada score and custom melodies
P5_SAMPLE_RATE = 16000                # audio sample rate (Hz)

P5_STR = {
    "title": {"en": "Quantum Four-Part Harmony Generator", "zh": "量子四部和声生成器"},
    "subtitle": {
        "en": "Enter a soprano melody → harmony rules are compiled into a Hamiltonian → a Qiskit solver returns alto, tenor and bass.",
        "zh": "输入女高音旋律 → 和声规则被编译为哈密顿量 → Qiskit 求解器返回中音、次中音与低音。"},
    "sec_melody": {"en": "1 · Melody", "zh": "1 · 旋律"},
    "sec_style": {"en": "2 · Style & solver", "zh": "2 · 风格与求解器"},
    "melody": {"en": "Melody", "zh": "选择旋律"},
    "opt_ode": {"en": "Ode to Joy (Beethoven)", "zh": "欢乐颂 (Beethoven)"},
    "opt_canada": {"en": "O Canada", "zh": "加拿大国歌 (O Canada)"},
    "opt_custom": {"en": "Custom notes (type or click)", "zh": "自定义音符（输入或点击）"},
    "opt_midi": {"en": "Upload a MIDI file", "zh": "上传 MIDI 文件"},
    "custom_hint": {
        "en": "Note names such as C4 F#4 Bb3; durations use beats or note values (C4:q., D4:e., E4:1.5). Use R:q for a rest; ':' in beats (E4:1.5, D4:0.5); '|' is ignored.",
        "zh": "音名如 C4 F#4 Bb3；冒号后可写时值（拍），如 E4:1.5、D4:0.5；'|' 会被忽略。"},
    "pal_dur": {"en": "Duration (beats)", "zh": "时值（拍）"},
    "pal_acc": {"en": "Accidental", "zh": "升降号"},
    "btn_undo": {"en": "Undo", "zh": "撤销"},
    "btn_clear": {"en": "Clear", "zh": "清空"},
    "key_label": {"en": "Key (major)", "zh": "调性（大调）"},
    "key_auto": {"en": "Auto-detect", "zh": "自动识别"},
    "ts_label": {"en": "Time signature", "zh": "拍号"},
    "pickup_label": {"en": "Pickup beats", "zh": "弱起拍数"},
    "style_label": {"en": "Style preset", "zh": "风格预设"},
    "variant_label": {"en": "Harmony choice", "zh": "????"},
    "variant_best": {"en": "Best sampled", "zh": "??????"},
    "strict_label": {"en": "Rule strictness", "zh": "规则权重倍数"},
    "solver_label": {"en": "Solver", "zh": "求解器"},
    "s_viterbi": {"en": "viterbi - classical baseline (exact)", "zh": "viterbi - 经典基准（精确）"},
    "s_qaoa": {"en": "qaoa - standard X mixer", "zh": "qaoa - 标准 X 混合器"},
    "s_xy": {"en": "xy_qaoa - constraint-preserving XY mixer", "zh": "xy_qaoa - 约束保持 XY 混合器"},
    "adv_title": {"en": "Advanced quantum settings (passed to the pipeline if it accepts them)",
                  "zh": "高级量子设置（流程支持时才会传入）"},
    "adv_p": {"en": "QAOA layers p", "zh": "QAOA 层数 p"},
    "adv_shots": {"en": "Shots", "zh": "采样次数"},
    "adv_window": {"en": "Window size (slots)", "zh": "窗口大小（和弦数）"},
    "adv_noisy": {"en": "Noisy fake backend", "zh": "带噪声的假后端"},
    "tempo_label": {"en": "Tempo (BPM)", "zh": "速度 (BPM)"},
    "btn_preview": {"en": "Preview melody", "zh": "试听旋律"},
    "btn_run": {"en": "Run harmonization", "zh": "运行和声生成"},
    "status_ready": {"en": "Ready.", "zh": "就绪。"},
    "status_running": {"en": "Compiling rules and running the solver...", "zh": "正在编译规则并运行求解器..."},
    "status_done": {"en": "Done in {sec:.1f} s.", "zh": "完成，用时 {sec:.1f} 秒。"},
    "status_error": {"en": "Failed - see the first tab.", "zh": "失败 - 请查看第一个标签页。"},
    "tab_score": {"en": "Score & audio", "zh": "乐谱与音频"},
    "tab_report": {"en": "Quantum report", "zh": "量子报告"},
    "tab_export": {"en": "Export", "zh": "导出"},
    "preview_title": {"en": "Melody preview", "zh": "旋律预览"},
    "parsed": {"en": "Parsed {n} notes · key {key} major · {ts} · {beats:g} beats",
               "zh": "已解析 {n} 个音符 · {key} 大调 · {ts} 拍 · 共 {beats:g} 拍"},
    "no_notes_preset": {"en": "This preset's notes come from the team pipeline (Part 1); nothing to preview here.",
                        "zh": "该预设的音符由团队流程（Part 1）提供，此处无法预览。"},
    "mock_banner": {
        "en": "MOCK BACKEND - a placeholder classical harmonizer is answering. These are NOT quantum results.",
        "zh": "MOCK 后端 - 当前由占位的经典和声器回答，这些不是量子结果。"},
    "warn_title": {"en": "Integration warnings", "zh": "集成警告"},
    "err_title": {"en": "Something went wrong", "zh": "出错了"},
    "audio_full": {"en": "Full chorale (S+A+T+B)", "zh": "完整四部和声"},
    "audio_sop": {"en": "Soprano only (your input)", "zh": "仅女高音（你的输入）"},
    "score_title": {"en": "Four-part harmony", "zh": "四部和声"},
    "viol_marked": {"en": "Notes shown in red take part in a reported rule violation.",
                    "zh": "红色音符参与了被报告的规则违规。"},
    "export_saved": {"en": "Files saved in", "zh": "文件已保存到"},
    "dl_midi": {"en": "Download MIDI", "zh": "下载 MIDI"},
    "dl_xml": {"en": "Download MusicXML", "zh": "下载 MusicXML"},
    "dl_wav": {"en": "Download WAV", "zh": "下载 WAV"},
    "dl_png": {"en": "Download score (PNG)", "zh": "下载乐谱 (PNG)"},
    "no_m21": {"en": "music21 is not installed - MIDI/MusicXML export is unavailable (pip install music21).",
               "zh": "未安装 music21 - 无法导出 MIDI/MusicXML（pip install music21）。"},
    "rep_title": {"en": "Quantum resource & performance report", "zh": "量子资源与性能报告"},
    "rep_na": {"en": "-", "zh": "-"},
    "rep_pipeline": {"en": "Pipeline", "zh": "调用的流程"},
    "rep_solver": {"en": "Solver", "zh": "求解器"},
    "rep_backend": {"en": "Backend", "zh": "后端"},
    "rep_qubits": {"en": "Qubits (max per window)", "zh": "量子比特数（单窗口最大）"},
    "rep_depth": {"en": "Circuit depth", "zh": "电路深度"},
    "rep_2q": {"en": "Two-qubit gates", "zh": "两比特门数量"},
    "rep_shots": {"en": "Shots", "zh": "采样次数"},
    "rep_layers": {"en": "QAOA layers p", "zh": "QAOA 层数 p"},
    "rep_runtime": {"en": "Solver runtime (s)", "zh": "求解耗时 (s)"},
    "rep_slots": {"en": "Chord slots", "zh": "和弦数"},
    "rep_viol": {"en": "Rule violations", "zh": "规则违规数"},
    "rep_energy": {"en": "Energy found", "zh": "所得能量"},
    "rep_classical": {"en": "Classical optimum (Viterbi)", "zh": "经典最优（Viterbi）"},
    "rep_ratio": {"en": "Approximation ratio", "zh": "近似比"},
    "rep_popt": {"en": "P(optimum) in samples", "zh": "采样中最优解概率"},
    "rep_feas": {"en": "Feasible samples", "zh": "可行样本比例"},
    "rep_viol_none": {"en": "No rule violations reported.", "zh": "未报告规则违规。"},
    "rep_viol_head": {"en": "Reported rule violations", "zh": "已报告的规则违规"},
    "rep_windows": {"en": "Per-window resources", "zh": "各窗口资源"},
    "rep_win_no": {"en": "Window", "zh": "窗口"},
    "rep_win_slots": {"en": "Slots", "zh": "和弦"},
    "rep_win_gap": {"en": "Gap", "zh": "差距"},
    "rep_chart": {"en": "Energy per window (lower is better)", "zh": "各窗口能量（越低越好）"},
    "rep_q_series": {"en": "Quantum solution", "zh": "量子解"},
    "rep_c_series": {"en": "Classical optimum", "zh": "经典最优"},
    "rep_circuit": {"en": "Circuit of the first window", "zh": "第一个窗口的电路"},
    "rep_hardware": {"en": "Recorded hardware run", "zh": "已记录的真机运行"},
    "rep_verdict": {"en": "Verdict from the pipeline", "zh": "流程给出的结论"},
    "rep_honest_title": {"en": "Honest assessment (Why quantum?)", "zh": "诚实评估（为什么用量子？）"},
    "rep_honest": {
        "en": ("Four-part harmony is solved exactly and efficiently by classical dynamic programming (Viterbi); "
               "this tool ships that solver as the baseline and does not claim a quantum advantage. "
               "Its contribution is the bridge: musical rules → penalty Hamiltonian → a swappable Qiskit solver, "
               "with errors you can hear and a constraint-preserving (XY-mixer) QAOA to study."),
        "zh": ("四部和声可以用经典动态规划（Viterbi）精确且高效地求解；本工具把它作为基准一并提供，"
               "并不声称量子优势。本项目的价值在于这座桥：音乐规则 → 惩罚哈密顿量 → 可替换的 Qiskit 求解器，"
               "错误能被听出来，并可研究约束保持（XY 混合器）QAOA。")},
}


def p5_t(key, **kw):
    """Look up a UI string in the language chosen by P5_LANG (falls back to English / the key itself)."""
    entry = P5_STR.get(key)
    if entry is None:
        return key
    s = entry.get(P5_LANG) or entry["en"]
    return s.format(**kw) if kw else s

In [2]:
# P5-01 Melody input: note names, validation, key guess, presets, MIDI upload
P5_LETTERS = "CDEFGAB"
P5_LETTER_PC = {"C": 0, "D": 2, "E": 4, "F": 5, "G": 7, "A": 9, "B": 11}
P5_KEY_FIFTHS = {"C": 0, "G": 1, "D": 2, "A": 3, "E": 4, "F": -1, "Bb": -2, "Eb": -3, "Ab": -4}
P5_SHARP_ORDER = "FCGDAEB"
P5_FLAT_ORDER = "BEADGCF"
P5_SOPRANO_SOFT = (60, 81)     # C4 .. A5: outside -> warning
P5_SOPRANO_HARD = (55, 88)     # outside -> error
P5_DUR_LETTERS = {"w": 4.0, "h": 2.0, "q": 1.0, "e": 0.5, "s": 0.25}

# Preset melodies owned by the UI. "text" is in the same syntax users type.
# O Canada follows the 28-bar score supplied for this project. Rests are preserved in the UI.
P5_PRESETS = {
    "ode_to_joy": {
        "label_key": "opt_ode", "title": "Ode to Joy", "key": "C", "time_sig": (4, 4), "pickup": 0.0, "tempo": 100,
        "text": ("E4 E4 F4 G4 | G4 F4 E4 D4 | C4 C4 D4 E4 | E4:1.5 D4:0.5 D4:2 | "
                 "E4 E4 F4 G4 | G4 F4 E4 D4 | C4 C4 D4 E4 | D4:1.5 C4:0.5 C4:2"),
    },
    "o_canada": {
        "label_key": "opt_canada", "title": "O Canada", "key": "F", "time_sig": (4, 4), "pickup": 0.0, "tempo": 80,
        "text": ("A4:h C5:q. C5:e | F4:h. G4:q | A4:q Bb4:q C5:q D5:q | G4:h. R:q | "
                 "A4:h Bb4:q. Bb4:e | C5:h. D5:q | E5:q E5:q D5:q D5:q | C5:h. G4:e. A4:s | "
                 "Bb4:q. A4:e G4:q A4:e. Bb4:s | C5:q. Bb4:e A4:q Bb4:e. C5:s | D5:q C5:q Bb4:q A4:q | "
                 "G4:h. G4:e. A4:s | Bb4:q. A4:e G4:q A4:e. Bb4:s | C5:q. Bb4:e A4:q A4:q | "
                 "G4:q C5:q C5:e Bb4:e A4:e Bb4:e | C5:h. R:q | A4:h C5:q. C5:e | F4:h. R:q | "
                 "Bb4:h D5:q. D5:e | G4:h. R:q | C5:h C#5:q. C#5:e | D5:q Bb4:q A4:q G4:q | "
                 "F4:h G4:h | A4:h. R:q | C5:h F5:q. F5:e | D5:q Bb4:q A4:q G4:q | C5:h E4:h | F4:w"),
    },
}


def p5_key_tonic_pc(key):
    """Pitch class (0-11) of a major-key tonic given as 'C', 'F#', 'Bb' ..."""
    alt = {"#": 1, "b": -1}.get(key[1:2], 0)
    return (P5_LETTER_PC[key[0].upper()] + alt) % 12


def p5_key_signature(key):
    """{letter: +1/-1} altered by the key signature of a major key."""
    n = P5_KEY_FIFTHS[key]
    if n > 0:
        return {L: 1 for L in P5_SHARP_ORDER[:n]}
    if n < 0:
        return {L: -1 for L in P5_FLAT_ORDER[:-n]}
    return {}


def p5_spell(midi, key="C"):
    """Spell a MIDI pitch as (letter, alteration, octave) using the key signature."""
    sig = p5_key_signature(key)
    pc = midi % 12
    for L in P5_LETTERS:                                   # 1) diatonic spelling
        alt = sig.get(L, 0)
        if (P5_LETTER_PC[L] + alt) % 12 == pc:
            return L, alt, (midi - alt) // 12 - 1
    for L in P5_LETTERS:                                   # 2) natural white key (needs a natural sign)
        if P5_LETTER_PC[L] == pc:
            return L, 0, midi // 12 - 1
    fifths = P5_KEY_FIFTHS[key]                            # 3) black key outside the key
    if fifths > 0:
        table = {1: ("C", 1), 3: ("D", 1), 6: ("F", 1), 8: ("G", 1), 10: ("A", 1)}
    elif fifths < 0:
        table = {1: ("C", 1), 3: ("E", -1), 6: ("G", -1), 8: ("A", -1), 10: ("B", -1)} if key == "F" else {1: ("D", -1), 3: ("E", -1), 6: ("G", -1), 8: ("A", -1), 10: ("B", -1)}
    else:
        table = {1: ("C", 1), 3: ("E", -1), 6: ("F", 1), 8: ("G", 1), 10: ("B", -1)}
    L, alt = table[pc]
    return L, alt, (midi - alt) // 12 - 1


def p5_note_name(midi, key="C"):
    """Human-readable name like 'F#4' or 'Bb3'."""
    L, alt, o = p5_spell(midi, key)
    return f"{L}{ {1: '#', -1: 'b', 0: ''}[alt] }{o}"


def p5_parse_dur(txt):
    """'1.5' / '1/2' / 'q' / 'e.' -> beats (quarter note = 1)."""
    txt = txt.strip().lower()
    m = p5_re.fullmatch(r"([whqes])(\.*)", txt)
    if m:
        d = P5_DUR_LETTERS[m.group(1)]
        add = d / 2
        for _ in m.group(2):
            d += add
            add /= 2
        return d
    if "/" in txt:
        a, b = txt.split("/", 1)
        return float(a) / float(b)
    return float(txt)


def p5_parse_melody_text(text):
    """Parse 'E4 E4 F4:1.5 ...' -> (notes, errors); notes = [(midi, beats), ...]."""
    notes, errors = [], []
    toks = [t for t in p5_re.split(r"[\s,;|]+", (text or "").strip()) if t]
    acc_val = {"#": 1, "♯": 1, "b": -1, "♭": -1, "-": -1}
    for k, tok in enumerate(toks, 1):
        rm = p5_re.fullmatch(r"(?:R|REST)(?::(\S+))?", tok, p5_re.I)
        if rm:
            try:
                beats = 1.0 if rm.group(1) is None else p5_parse_dur(rm.group(1))
            except (ValueError, ZeroDivisionError):
                errors.append(f"#{k} '{tok}': cannot read the rest duration")
                continue
            if not (0 < beats <= 8):
                errors.append(f"#{k} '{tok}': duration must be between 0 and 8 beats")
                continue
            notes.append((None, float(beats)))
            continue
        m = p5_re.fullmatch(r"([A-Ga-g])([#♯b♭-]{0,2})(-?\d)(?::(\S+))?", tok)
        if not m:
            errors.append(f"#{k} '{tok}' is not a note (examples: C4, F#4, Bb3, E4:1.5)")
            continue
        alt = sum(acc_val[c] for c in m.group(2))
        midi = 12 * (int(m.group(3)) + 1) + P5_LETTER_PC[m.group(1).upper()] + alt
        try:
            beats = 1.0 if m.group(4) is None else p5_parse_dur(m.group(4))
        except (ValueError, ZeroDivisionError):
            errors.append(f"#{k} '{tok}': cannot read the duration '{m.group(4)}'")
            continue
        if not (0 < beats <= 8):
            errors.append(f"#{k} '{tok}': duration must be between 0 and 8 beats")
            continue
        notes.append((midi, float(beats)))
    return notes, errors


def p5_validate_melody(notes, time_sig=(4, 4)):
    """Return (errors, warnings) for a list of (midi, beats)."""
    errors, warnings = [], []
    if sum(m is not None for m, _ in notes) < 2:
        errors.append("Enter at least 2 pitched notes.")
    if len(notes) > P5_MAX_NOTES:
        errors.append(f"Too many notes ({len(notes)}); the limit is {P5_MAX_NOTES}.")
    lo, hi = P5_SOPRANO_HARD
    slo, shi = P5_SOPRANO_SOFT
    for i, (m, _) in enumerate(notes, 1):
        if m is None:
            continue
        if not lo <= m <= hi:
            errors.append(f"Note #{i} ({p5_note_name(m)}) is outside the supported soprano range.")
        elif not slo <= m <= shi:
            warnings.append(f"Note #{i} ({p5_note_name(m)}) is outside the usual soprano range C4-A5.")
    for i in range(1, len(notes)):
        if notes[i][0] is not None and notes[i - 1][0] is not None and abs(notes[i][0] - notes[i - 1][0]) > 12:
            warnings.append(f"Leap larger than an octave between notes #{i} and #{i + 1}.")
    return errors, warnings


def p5_guess_key(notes):
    """Best-fitting major key among C G D A E F Bb Eb Ab (diatonic coverage, tonic-at-the-end bonus)."""
    pcs = [m % 12 for m, _ in notes if m is not None]
    if not pcs:
        return "C"
    best, best_score = "C", -1.0
    for name, fifths in P5_KEY_FIFTHS.items():
        tonic = p5_key_tonic_pc(name)
        scale = {(tonic + i) % 12 for i in (0, 2, 4, 5, 7, 9, 11)}
        score = sum(p in scale for p in pcs) / max(1, len(pcs))
        score += 0.03 * (pcs[-1] == tonic) + 0.01 * (pcs[0] == tonic) - 0.001 * abs(fifths)
        if score > best_score:
            best, best_score = name, score
    return best


def p5_make_melody(notes, key=None, time_sig=(4, 4), pickup=0.0, title="", preset=None, tempo=90, source="text"):
    """Normalized melody dict handed to the pipeline (see README: integration contract)."""
    return {
        "title": title, "preset": preset, "source": source,
        "notes": [(None if m is None else int(m), float(b)) for m, b in notes] if notes else None,
        "key": key or (p5_guess_key(notes) if notes else None), "mode": "major",
        "time_sig": tuple(time_sig), "pickup": float(pickup), "tempo": int(tempo),
    }


def p5_preset_melody(preset_id):
    """Melody dict for a preset id, resolving canonical Part 1 presets when available."""
    spec = P5_PRESETS[preset_id]
    notes = None
    if spec["text"]:
        notes, errs = p5_parse_melody_text(spec["text"])
        if errs:
            raise ValueError("; ".join(errs))
    return p5_make_melody(notes, key=spec["key"], time_sig=spec["time_sig"], pickup=spec["pickup"],
                          title=spec["title"], preset=preset_id, tempo=spec["tempo"], source="preset")


def p5_parse_midi_bytes(data, title="MIDI upload"):
    """Read a monophonic melody from MIDI bytes (needs music21). Returns a melody dict."""
    if p5_m21 is None:
        raise RuntimeError("music21 is required to read MIDI files (pip install music21).")
    with p5_tmp.TemporaryDirectory() as d:
        path = p5_os.path.join(d, "in.mid")
        with open(path, "wb") as f:
            f.write(data)
        s = p5_m21.converter.parse(path)
    flat = s.flatten()
    items = []
    for el in flat.notes:
        top = max(p.midi for p in el.pitches)                 # chord -> keep the top pitch
        items.append((float(el.offset), top, float(el.quarterLength)))
    items.sort()
    notes, end = [], -1.0
    for off, m, ql in items:                                   # drop overlapping notes
        if off >= end - 1e-6:
            notes.append((m, max(0.25, round(ql * 4) / 4)))
            end = off + ql
    ts, key = (4, 4), None
    for t in flat.getElementsByClass("TimeSignature"):
        ts = (int(t.numerator), int(t.denominator))
        break
    for k in flat.getElementsByClass("KeySignature"):
        try:
            cand = {0: "C", 1: "G", 2: "D", 3: "A", 4: "E", -1: "F", -2: "Bb", -3: "Eb", -4: "Ab"}[int(k.sharps)]
            key = cand
        except KeyError:
            pass
        break
    return p5_make_melody(notes, key=key, time_sig=ts, title=title, source="midi")

In [3]:
# P5-02 Data model: P5Score, measure splitting, normalizing whatever the pipeline returns
# Voice order everywhere in Part 5 follows the team contract: pitches = (bass, tenor, alto, soprano).
P5_VOICE_NAMES = ("Bass", "Tenor", "Alto", "Soprano")
P5_RANGES = {0: (40, 64), 1: (48, 69), 2: (55, 74), 3: (60, 81)}     # soft SATB ranges (E2-E4, C3-A4, G3-D5, C4-A5)


@p5_dataclass
class P5Slot:
    pitches: tuple                 # (bass, tenor, alto, soprano) as MIDI numbers
    beats: float = 1.0             # duration in quarter notes
    chord: str = ""                # optional Roman-numeral label such as "V7"
    soprano_rest: bool = False      # preserve a written soprano rest through rendering/export


@p5_dataclass
class P5Score:
    slots: list
    key: str = "C"                 # major-key tonic: C G D A E F Bb Eb Ab
    time_sig: tuple = (4, 4)
    tempo: int = 90
    pickup: float = 0.0            # beats in an incomplete first measure (0 = none)
    title: str = ""
    warnings: list = p5_field(default_factory=list)

    @property
    def bar_len(self):
        return self.time_sig[0] * 4.0 / self.time_sig[1]


def p5_split_measures(beats_list, bar_len, pickup=0.0):
    """Group slots into measures. Each measure is a list of pieces (slot_index, beats, tie_in, tie_out);
    a note that crosses a barline is split into tied pieces."""
    eps = 1e-6
    measures, cur = [], []
    room = pickup if pickup > eps else bar_len
    for i, b in enumerate(beats_list):
        left, tie_in = float(b), False
        while left > eps:
            piece = min(left, room)
            tie_out = (left - piece) > eps
            cur.append((i, piece, tie_in, tie_out))
            left -= piece
            room -= piece
            tie_in = True
            if room <= eps:
                measures.append(cur)
                cur, room = [], bar_len
    if cur:
        measures.append(cur)
    return measures


def p5_is_m21_stream(obj):
    return p5_m21 is not None and isinstance(obj, p5_m21.stream.Stream)


def p5_voicing_pitches(v):
    """Pitches (B,T,A,S) from a tuple, a dict or an object with .pitches."""
    if isinstance(v, dict):
        v = v.get("pitches", v)
    elif hasattr(v, "pitches"):
        v = v.pitches
    p = tuple(int(round(float(x))) for x in v)
    if len(p) != 4:
        raise ValueError(f"A voicing needs 4 pitches (bass, tenor, alto, soprano); got {len(p)}.")
    return p


def p5_from_m21(stream, melody):
    """Stream with >=4 parts (S,A,T,B top to bottom) or chords of 4 notes -> P5Score."""
    parts = list(stream.parts) if hasattr(stream, "parts") else []
    slots = []
    if len(parts) >= 4:
        sop = list(parts[0].flatten().notes)
        others = [p.flatten().notes for p in parts[1:4]]
        for n in sop:
            top = max(x.midi for x in n.pitches)
            row = [top]
            for notes in others:
                el = notes.getElementAtOrBefore(n.offset) if hasattr(notes, "getElementAtOrBefore") else None
                row.append(max(x.midi for x in el.pitches) if el is not None else top)
            s, a, t, b = row
            slots.append(P5Slot((b, t, a, s), float(n.quarterLength)))
    else:
        for n in stream.flatten().notes:
            ps = sorted(x.midi for x in n.pitches)
            if len(ps) < 4:
                raise ValueError("music21 result needs 4 parts or 4-note chords.")
            slots.append(P5Slot(tuple(ps[:3] + [ps[-1]]), float(n.quarterLength)))
    return slots


def p5_normalize_result(result, melody=None):
    """Accept P5Score | dict{'voicings',...} | list of voicings | music21 Stream  ->  P5Score."""
    melody = melody or {}
    if isinstance(result, P5Score):
        return result
    meta = {"key": melody.get("key") or "C", "time_sig": tuple(melody.get("time_sig") or (4, 4)),
            "pickup": float(melody.get("pickup") or 0.0), "tempo": int(melody.get("tempo") or 90),
            "title": melody.get("title") or ""}
    notes = melody.get("notes")
    durations = chords = None
    warnings = []
    if p5_is_m21_stream(result):
        slots = p5_from_m21(result, melody)
    else:
        if isinstance(result, dict):
            voicings = result.get("voicings", result.get("slots"))
            durations, chords = result.get("durations"), result.get("chords")
            for k in meta:
                if result.get(k) is not None:
                    meta[k] = tuple(result[k]) if k == "time_sig" else result[k]
        else:
            voicings = result
        if voicings is None:
            raise ValueError("Pipeline result has no 'voicings'. Return a list of (bass, tenor, alto, soprano) "
                             "tuples, a dict with 'voicings', or a 4-part music21 Stream.")
        slots = []
        for i, v in enumerate(voicings):
            p = p5_voicing_pitches(v)
            if p[0] > p[3]:                                   # looks like (S, A, T, B)
                p = tuple(reversed(p))
                if i == 0:
                    warnings.append("Voicings looked like (soprano, alto, tenor, bass); reversed to (bass, tenor, alto, soprano).")
            chord = chords[i] if chords and i < len(chords) else (getattr(v, "chord", "") if not isinstance(v, (tuple, list, dict)) else (v.get("chord", "") if isinstance(v, dict) else ""))
            if durations and i < len(durations):
                b = float(durations[i])
            elif notes and len(notes) == len(voicings):
                b = float(notes[i][1])
            else:
                b = 1.0
            slots.append(P5Slot(p, b, str(chord or "")))
    if notes and len(notes) == len(slots):                    # durations from the input melody
        for s, (_, b) in zip(slots, notes):
            if s.beats == 1.0 and b != 1.0 and not durations:
                s.beats = float(b)
    sc = P5Score(slots=slots, warnings=warnings, **meta)
    sc.warnings += p5_sanity(sc, melody)
    return sc


def p5_sanity(sc, melody=None, limit=5):
    """Cheap integration checks (not harmony rules): soprano preserved, voice order, SATB ranges."""
    w = []
    notes = (melody or {}).get("notes")
    if notes and len(notes) == len(sc.slots):
        bad = [i + 1 for i, ((m, _), s) in enumerate(zip(notes, sc.slots)) if m is not None and s.pitches[3] != m]
        if bad:
            w.append(f"Soprano differs from the input melody at slot(s) {bad[:8]} - the solver must keep the soprano fixed.")
    elif notes:
        w.append(f"Input melody has {len(notes)} notes but the result has {len(sc.slots)} slots.")
    order = [i + 1 for i, s in enumerate(sc.slots) if not (s.pitches[0] <= s.pitches[1] <= s.pitches[2] <= s.pitches[3])]
    if order:
        w.append(f"Voices are not in bass ≤ tenor ≤ alto ≤ soprano order at slot(s) {order[:8]}.")
    for v in range(4):
        lo, hi = P5_RANGES[v]
        out = [i + 1 for i, s in enumerate(sc.slots) if not lo <= s.pitches[v] <= hi]
        if out:
            w.append(f"{P5_VOICE_NAMES[v]} leaves its usual range at slot(s) {out[:8]}.")
    return w[:limit]

In [4]:
# P5-03 Export: P5Score -> music21 -> MIDI / MusicXML, saving files, download links
P5_VOICE_ORDER_TOP_DOWN = (3, 2, 1, 0)       # score order: soprano, alto, tenor, bass
P5_RED = "#d62728"


def p5_m21_name(midi, key="C"):
    """Pitch name in music21 syntax ('F#4', 'B-3'), spelled for the key."""
    L, alt, o = p5_spell(midi, key)
    acc = {1: "#", -1: "-", 0: ""}[alt]
    return f"{L}{acc}{o}"


def p5_to_m21(sc, marks=None):
    """Build a 4-part music21 Score (open score, with measures, ties, chord labels under the bass).
    marks: {slot_index: set(voice indices) or None(all voices)} -> coloured red."""
    if p5_m21 is None:
        raise RuntimeError(p5_t("no_m21"))
    m21 = p5_m21
    marks = marks or {}
    m21_key = sc.key.replace("b", "-")
    groups = p5_split_measures([s.beats for s in sc.slots], sc.bar_len, sc.pickup)
    classes = {3: m21.instrument.Soprano, 2: m21.instrument.Alto, 1: m21.instrument.Tenor, 0: m21.instrument.Bass}
    clefs = {3: m21.clef.TrebleClef, 2: m21.clef.TrebleClef, 1: m21.clef.Treble8vbClef, 0: m21.clef.BassClef}
    score = m21.stream.Score()
    score.metadata = m21.metadata.Metadata()
    score.metadata.title = sc.title or "Quantum four-part harmony"
    for v in P5_VOICE_ORDER_TOP_DOWN:
        part = m21.stream.Part(id=P5_VOICE_NAMES[v])
        part.partName = P5_VOICE_NAMES[v]
        for mi, pieces in enumerate(groups):
            meas = m21.stream.Measure(number=mi if sc.pickup > 1e-6 else mi + 1)
            if mi == 0:
                meas.append(classes[v]())
                meas.append(clefs[v]())
                meas.append(m21.key.Key(m21_key))
                meas.append(m21.meter.TimeSignature(f"{sc.time_sig[0]}/{sc.time_sig[1]}"))
                if v == 3:
                    meas.append(m21.tempo.MetronomeMark(number=sc.tempo))
                if sc.pickup > 1e-6:
                    meas.paddingLeft = sc.bar_len - sc.pickup
            for si, beats, tie_in, tie_out in pieces:
                slot = sc.slots[si]
                if v == 3 and getattr(slot, "soprano_rest", False):
                    n = m21.note.Rest()
                else:
                    n = m21.note.Note(p5_m21_name(slot.pitches[v], sc.key))
                n.quarterLength = beats
                if tie_in and tie_out:
                    n.tie = m21.tie.Tie("continue")
                elif tie_in:
                    n.tie = m21.tie.Tie("stop")
                elif tie_out:
                    n.tie = m21.tie.Tie("start")
                if si in marks and (marks[si] is None or v in marks[si]):
                    n.style.color = P5_RED
                if v == 0 and slot.chord and not tie_in:
                    n.addLyric(slot.chord)
                meas.append(n)
            if mi == len(groups) - 1:
                meas.rightBarline = "final"
            part.append(meas)
        score.insert(0, part)
    return score


def p5_m21_bytes(sc, fmt, marks=None):
    """fmt = 'midi' or 'musicxml' -> file bytes."""
    score = p5_to_m21(sc, marks)
    with p5_tmp.TemporaryDirectory() as d:
        path = score.write(fmt, fp=p5_os.path.join(d, "out." + ("mid" if fmt == "midi" else "musicxml")))
        with open(path, "rb") as f:
            return f.read()


def p5_safe_stem(title):
    stem = p5_re.sub(r"[^A-Za-z0-9_-]+", "_", title or "harmony").strip("_") or "harmony"
    return f"{stem}_{p5_dt.datetime.now():%Y%m%d_%H%M%S}"


def p5_save_all(sc, marks=None, score_png=None, wav=None, out_dir=None):
    """Write .mid / .musicxml / .wav / .png into P5_OUT_DIR. Returns {label: (path, bytes, mime)} and errors."""
    out_dir = out_dir or P5_OUT_DIR
    p5_os.makedirs(out_dir, exist_ok=True)
    stem = p5_safe_stem(sc.title)
    files, errors = {}, []
    jobs = [("midi", ".mid", "audio/midi", lambda: p5_m21_bytes(sc, "midi", marks)),
            ("xml", ".musicxml", "application/vnd.recordare.musicxml+xml", lambda: p5_m21_bytes(sc, "musicxml", marks)),
            ("wav", ".wav", "audio/wav", lambda: wav if wav is not None else p5_synth_wav(sc)),
            ("png", ".png", "image/png", lambda: score_png)]
    for label, ext, mime, make in jobs:
        try:
            data = make()
            if data is None:
                continue
            path = p5_os.path.join(out_dir, stem + ext)
            with open(path, "wb") as f:
                f.write(data)
            files[label] = (path, data, mime)
        except Exception as e:
            errors.append(f"{label}: {type(e).__name__}: {e}")
    return files, errors


def p5_download_links_html(files):
    """Base64 download buttons (work in JupyterLab / classic Notebook; the saved paths are the fallback)."""
    names = {"midi": "dl_midi", "xml": "dl_xml", "wav": "dl_wav", "png": "dl_png"}
    style = ("display:inline-block;margin:4px 8px 4px 0;padding:6px 12px;border-radius:6px;"
             "background:#1565c0;color:#fff;text-decoration:none;font-family:system-ui,Arial,sans-serif;font-size:13px")
    out = []
    for label, (path, data, mime) in files.items():
        b64 = p5_b64.b64encode(data).decode("ascii")
        fname = p5_html.escape(p5_os.path.basename(path))
        out.append(f'<a style="{style}" download="{fname}" href="data:{mime};base64,{b64}">{p5_html.escape(p5_t(names[label]))}</a>')
    return "".join(out)

In [5]:
# P5-04 Audio: offline additive synthesis -> WAV bytes -> <audio> player
# A .mid file cannot be played by a browser <audio> tag, so the notebook synthesizes the chorale itself
# with numpy (no soundfont / FluidSynth needed). MIDI is still exported for DAWs and MuseScore.
P5_HARMONICS = (1.0, 0.5, 0.33, 0.22, 0.12, 0.07)       # organ/choir-like spectrum
P5_VOICE_GAIN = {0: 1.0, 1: 0.8, 2: 0.8, 3: 0.95}


def p5_midi_hz(m):
    return 440.0 * 2.0 ** ((m - 69) / 12.0)


def p5_synth_wav(sc, voices=(0, 1, 2, 3), tempo=None, sr=None):
    """Synthesize the chosen voices (indices into pitches) and return mono 16-bit WAV bytes."""
    sr = sr or P5_SAMPLE_RATE
    tempo = tempo or sc.tempo or 90
    spb = 60.0 / tempo
    tail = 0.25
    total = sum(s.beats for s in sc.slots) * spb + tail + 0.3
    out = p5_np.zeros(int(total * sr) + 1)
    t0 = 0.0
    for s in sc.slots:
        dur = s.beats * spb
        n = int((dur + tail) * sr)
        tt = p5_np.arange(n) / sr
        env = p5_np.ones(n)
        att = max(1, int(0.03 * sr))
        env[:att] = p5_np.linspace(0.0, 1.0, att)
        rel0 = int(dur * 0.94 * sr)
        env[rel0:] = p5_np.linspace(1.0, 0.0, n - rel0) ** 2
        i0 = int(t0 * sr)
        for v in voices:
            if v == 3 and getattr(s, "soprano_rest", False):
                continue
            f = p5_midi_hz(s.pitches[v])
            wave = sum(a * p5_np.sin(2 * p5_math.pi * f * (k + 1) * tt)
                       for k, a in enumerate(P5_HARMONICS) if f * (k + 1) < sr * 0.45)
            seg = wave * env * P5_VOICE_GAIN[v]
            out[i0:i0 + n] += seg[: len(out) - i0]
        t0 += dur
    peak = float(p5_np.max(p5_np.abs(out))) or 1.0
    out = p5_np.tanh(1.4 * out / peak) / p5_np.tanh(1.4) * 0.85
    pcm = (out * 32767).astype("<i2")
    buf = p5_io.BytesIO()
    with p5_wave.open(buf, "wb") as w:
        w.setnchannels(1)
        w.setsampwidth(2)
        w.setframerate(sr)
        w.writeframes(pcm.tobytes())
    return buf.getvalue()


def p5_audio_html(wav_bytes, label=""):
    """Self-contained <audio> element (data URI), visible in Jupyter, VS Code and nbviewer."""
    b64 = p5_b64.b64encode(wav_bytes).decode("ascii")
    return ('<div style="font-family:system-ui,Arial,sans-serif;font-size:13px;margin:6px 0;color:#222">'
            f'<div style="margin-bottom:2px"><b>{p5_html.escape(label)}</b></div>'
            f'<audio controls preload="auto" src="data:audio/wav;base64,{b64}"></audio></div>')

In [6]:
# P5-05 Score renderer: grand staff drawn with matplotlib (offline, no MuseScore / LilyPond needed)
P5_SYS_H = 23.5                                   # vertical size of one system (in staff spaces)
P5_SHARP_STEPS = ([38, 35, 39, 36, 33, 37, 34], [24, 21, 25, 22, 19, 23, 20])   # (treble, bass): F C G D A E B
P5_FLAT_STEPS = ([34, 37, 33, 36, 32, 35, 31], [20, 23, 19, 22, 18, 21, 17])    # B E A D G C F
P5_TREBLE_CLEF = [(0.55, -1.35), (0.75, -1.75), (1.2, -1.9), (1.55, -1.55), (1.55, -0.6), (1.45, 1.0), (1.45, 2.6),
                  (1.55, 4.2), (1.7, 5.2), (2.0, 5.5), (2.35, 5.0), (2.35, 4.1), (1.95, 3.3), (1.2, 2.9), (0.45, 2.4),
                  (0.2, 1.5), (0.5, 0.55), (1.15, 0.25), (1.9, 0.55), (2.3, 1.2), (2.05, 1.9), (1.45, 2.1), (0.95, 1.7),
                  (0.95, 1.2), (1.3, 0.95), (1.6, 1.2)]
P5_BASS_CLEF = [(0.7, 3.0), (0.95, 3.75), (1.6, 3.95), (2.25, 3.45), (2.35, 2.5), (1.9, 1.4), (1.0, 0.6), (0.35, 0.1)]


def p5_spline(points, n=14):
    """Catmull-Rom spline through the points -> dense polyline (N x 2 array)."""
    P = p5_np.array(points, float)
    P = p5_np.vstack([P[0], P, P[-1]])
    out = []
    for i in range(1, len(P) - 2):
        p0, p1, p2, p3 = P[i - 1], P[i], P[i + 1], P[i + 2]
        for t in p5_np.linspace(0, 1, n, endpoint=False):
            out.append(0.5 * (2 * p1 + (-p0 + p2) * t + (2 * p0 - 5 * p1 + 4 * p2 - p3) * t ** 2
                              + (-p0 + 3 * p1 - 3 * p2 + p3) * t ** 3))
    out.append(P[-2])
    return p5_np.array(out)


def p5_step_y(step):
    """Vertical position (staff spaces) of a diatonic step; E4 (step 30) is the bottom treble line, y = 0."""
    return (step - 30) / 2.0


def p5_ledger_ys(y, treble):
    if treble:
        low = list(range(int(p5_math.ceil(y)), 0)) if y <= -1 else []
        return list(range(5, int(p5_math.floor(y)) + 1)) + low
    low = list(range(int(p5_math.ceil(y)), -6)) if y <= -7 else []
    return low + (list(range(-1, int(p5_math.floor(y)) + 1)) if y >= -1 else [])


def p5_draw_clefs(ax, x, oy, pt):
    xy = p5_spline(P5_TREBLE_CLEF)
    ax.plot(x + xy[:, 0], oy + xy[:, 1], color="black", lw=0.17 * pt, solid_capstyle="round", zorder=3)
    ax.add_patch(p5_Circle((x + 0.58, oy - 1.3), 0.2, color="black", zorder=3))
    xy = p5_spline(P5_BASS_CLEF)
    by = oy - 6
    ax.plot(x + xy[:, 0], by + xy[:, 1], color="black", lw=0.2 * pt, solid_capstyle="round", zorder=3)
    ax.add_patch(p5_Circle((x + 0.7, by + 3.0), 0.4, color="black", zorder=3))
    for dy in (3.5, 2.5):
        ax.add_patch(p5_Circle((x + 3.0, by + dy), 0.22, color="black", zorder=3))


def p5_engrave_png(sc, marks=None, windows=None, title=None, width_in=13.5, dpi=110):
    """Render the 4-part score as PNG bytes.
    marks   {slot_index: set(voice idx) | None}  -> notes drawn in red (rule violations)
    windows [(start, end_exclusive), ...]        -> shaded bands labelled W1, W2, ... (QAOA windows)"""
    key, sig = sc.key, p5_key_signature(sc.key)
    n_acc = len(sig)
    marks = marks or {}
    groups = p5_split_measures([s.beats for s in sc.slots], sc.bar_len, sc.pickup)

    plan = {}                                            # accidentals needed: (measure, piece, voice) -> alteration
    for mi, pieces in enumerate(groups):
        state = {}
        for pi, (si, b, tie_in, tie_out) in enumerate(pieces):
            if tie_in:
                continue
            for v in range(4):
                if v == 3 and getattr(sc.slots[si], "soprano_rest", False):
                    continue
                L, alt, o = p5_spell(sc.slots[si].pitches[v], key)
                if alt != state.get((L, o), sig.get(L, 0)):
                    plan[(mi, pi, v)] = alt
                    state[(L, o)] = alt

    wmax = 88.0
    hdr = lambda first: 4.6 + 1.25 * n_acc + (3.4 if first else 0.0)
    meas = []
    for mi, pieces in enumerate(groups):
        ws = [2.6 + 1.8 * min(b, 4.0) ** 0.75 + (1.5 if any((mi, pi, v) in plan for v in range(4)) else 0.0)
              for pi, (si, b, _, _) in enumerate(pieces)]
        meas.append((ws, sum(ws) + 2.0))
    systems, cur, cw = [], [], 0.0
    for mi, (ws, w) in enumerate(meas):
        if cur and cw + w > wmax - hdr(len(systems) == 0):
            systems.append((cur, cw))
            cur, cw = [], 0.0
        cur.append(mi)
        cw += w
    if cur:
        systems.append((cur, cw))
    n_sys = len(systems)

    x0, x1 = -1.0, wmax + 2.0
    ytop, ybot = 13.0, -(n_sys - 1) * P5_SYS_H - 13.2
    fig = p5_Figure(figsize=(width_in, width_in * (ytop - ybot) / (x1 - x0)), dpi=dpi, facecolor="white")
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(x0, x1)
    ax.set_ylim(ybot, ytop)
    ax.axis("off")
    pt = width_in * 72.0 / (x1 - x0)                     # points per staff space
    lw_staff, lw_stem = 0.09 * pt, 0.11 * pt
    Y = lambda y, s: y - s * P5_SYS_H
    ax.text(0, ytop - 0.4, title if title is not None else (sc.title or p5_t("score_title")),
            fontsize=1.3 * pt, fontweight="bold", va="top", color="#222")

    pos = {}                                             # slot -> [(system, x_center, width)]
    prev_head = {}                                       # (slot, voice) -> (x, y, system, stem_up) for ties
    last_measure = len(groups) - 1
    for s, (mlist, cwid) in enumerate(systems):
        hd = hdr(s == 0)
        stretch = 1.0 if s == n_sys - 1 else min(1.6, (wmax - hd) / cwid)
        x_end = hd + cwid * stretch
        for k in range(5):
            for base in (0, -6):
                ax.plot([0, x_end], [Y(base + k, s)] * 2, color="black", lw=lw_staff, solid_capstyle="butt", zorder=1)
        ax.plot([0, 0], [Y(4, s), Y(-6, s)], color="black", lw=lw_stem, zorder=1)
        p5_draw_clefs(ax, 0.5, Y(0, s), pt)
        xk = 4.4
        steps = (P5_SHARP_STEPS if P5_KEY_FIFTHS[key] > 0 else P5_FLAT_STEPS)
        glyph = "♯" if P5_KEY_FIFTHS[key] > 0 else "♭"
        for i in range(n_acc):
            for st in (steps[0][i], steps[1][i]):
                ax.text(xk, Y(p5_step_y(st), s) + (0.35 if glyph == "♭" else 0), glyph, fontsize=3.4 * pt,
                        ha="center", va="center", zorder=3)
            xk += 1.25
        if s == 0:
            for (cy, num, den) in ((3, sc.time_sig[0], sc.time_sig[1]), (1, None, None), (-3, sc.time_sig[0], sc.time_sig[1]), (-5, None, None)):
                txt = str(num) if num is not None else str(sc.time_sig[1])
                ax.text(xk + 1.0, Y(cy, s), txt, fontsize=2.6 * pt, fontweight="bold", ha="center", va="center", zorder=3)
        x = hd
        for mi in mlist:
            ws, _ = meas[mi]
            x += 1.0
            for pi, (si, beats, tie_in, tie_out) in enumerate(groups[mi]):
                w = ws[pi] * stretch
                has_acc = any((mi, pi, v) in plan for v in range(4))
                xc = x + w / 2 + (0.6 if has_acc else 0.0)
                pos.setdefault(si, []).append((s, xc, w))
                marked = si in marks
                if marked:
                    ax.add_patch(p5_Rect((x - 0.1, Y(-10.8, s)), w + 0.2, 19.6, facecolor="#ffe3e3", edgecolor="none", zorder=0.5))
                if not tie_in:
                    ax.text(xc, Y(8.0, s), str(si + 1), fontsize=0.8 * pt, color="#999", ha="center", va="center")
                    if sc.slots[si].chord:
                        ax.text(xc, Y(-11.4, s), sc.slots[si].chord, fontsize=1.0 * pt, color="#444", ha="center", va="center")
                p5_engrave_piece(ax, sc, plan, mi, pi, si, beats, tie_in, tie_out, xc, s, Y, pt, marks, prev_head, lw_stem)
                x += w
            x += 1.0
            thick = (mi == last_measure)
            ax.plot([x, x], [Y(4, s), Y(-6, s)], color="black", lw=(0.4 if thick else 0.11) * pt, zorder=1)
            if thick:
                ax.plot([x - 0.5, x - 0.5], [Y(4, s), Y(-6, s)], color="black", lw=0.11 * pt, zorder=1)

    for k, (a, b) in enumerate(windows or []):
        for s in range(n_sys):
            spans = [(xc - w / 2, xc + w / 2) for si in range(a, b) for (ss, xc, w) in pos.get(si, []) if ss == s]
            if spans:
                xa, xb = min(p[0] for p in spans), max(p[1] for p in spans)
                ax.add_patch(p5_Rect((xa - 0.1, Y(-12.6, s)), xb - xa + 0.2, 23.0, facecolor="#e3f0ff" if k % 2 == 0 else "#e8f5e9",
                                     edgecolor="none", zorder=0))
                ax.text(xa + 0.2, Y(9.7, s), f"W{k + 1}", fontsize=0.9 * pt, color="#1565c0", va="center")
    buf = p5_io.BytesIO()
    p5_Canvas(fig).print_png(buf)
    return buf.getvalue()


def p5_engrave_piece(ax, sc, plan, mi, pi, si, beats, tie_in, tie_out, xc, s, Y, pt, marks, prev_head, lw_stem):
    """Draw the four noteheads (with stems, flags, dots, ledger lines, accidentals, ties) of one piece."""
    key = sc.key
    slot = sc.slots[si]
    base = 2.0 ** p5_math.floor(p5_math.log2(beats + 1e-9))
    ratio = beats / base
    dots = 1 if abs(ratio - 1.5) < 0.05 else (2 if abs(ratio - 1.75) < 0.05 else 0)
    flags = 0 if base > 0.5 + 1e-9 else int(round(-p5_math.log2(base)))
    whole = beats >= 4.0 - 1e-6
    open_head = beats >= 2.0 - 1e-6
    for vu, vd, mid, treble in ((3, 2, 2.0, True), (1, 0, -4.0, False)):
        info = {}
        for v in (vu, vd):
            if v == 3 and getattr(slot, "soprano_rest", False):
                continue
            L, alt, o = p5_spell(slot.pitches[v], key)
            step = 7 * o + P5_LETTERS.index(L)
            info[v] = (step, p5_step_y(step))
        dx = {vu: 0.0, vd: 0.0}
        unison = vu in info and vd in info and info[vu][0] == info[vd][0]
        if vu in info and vd in info and abs(info[vu][0] - info[vd][0]) == 1:
            dx[vu if info[vu][0] > info[vd][0] else vd] = 1.15
        accs = []
        if vu == 3 and getattr(slot, "soprano_rest", False):
            rest_col = "#d62728" if si in marks else "black"
            rest_path = p5_Path([(xc - 0.2, Y(3.0, s)), (xc + 0.55, Y(2.2, s)),
                                 (xc - 0.25, Y(1.45, s)), (xc + 0.35, Y(0.6, s)),
                                 (xc + 0.05, Y(0.15, s)), (xc + 0.75, Y(-0.35, s)),
                                 (xc + 0.15, Y(-0.5, s))],
                                [p5_Path.MOVETO, p5_Path.CURVE3, p5_Path.CURVE3,
                                 p5_Path.CURVE3, p5_Path.CURVE3, p5_Path.CURVE3, p5_Path.CURVE3])
            ax.add_patch(p5_PathPatch(rest_path, facecolor="none", edgecolor=rest_col,
                                      lw=0.28 * pt, zorder=4))
        for v in (vu, vd):
            if v not in info:
                continue
            step, y = info[v]
            red = si in marks and (marks[si] is None or v in marks[si])
            col = "#d62728" if red else "black"
            up = v in (3, 1)
            if not (unison and v == vd):                                  # one head for a unison
                xh = xc + dx[v]
                for ly in p5_ledger_ys(y, treble):
                    ax.plot([xh - 1.05, xh + 1.05], [Y(ly, s)] * 2, color="black", lw=0.09 * pt, zorder=2)
                ax.add_patch(p5_Ellipse((xh, Y(y, s)), 1.7 if whole else 1.28, 0.95 if whole else 0.92,
                                        angle=0 if whole else 22, facecolor="white" if open_head else col,
                                        edgecolor=col, lw=0.11 * pt, zorder=4))
                if dots:
                    for d in range(dots):
                        ax.add_patch(p5_Circle((xh + 1.0 + 0.55 * d, Y(y + (0.5 if step % 2 == 0 else 0.0), s)), 0.17, color=col, zorder=4))
            xh = xc + dx[v]
            if not whole:
                sx = xh + 0.6 if up else xh - 0.6
                ye = max(y + 3.3, mid) if up else min(y - 3.3, mid)
                ax.plot([sx, sx], [Y(y + (0.1 if up else -0.1), s), Y(ye, s)], color=col, lw=lw_stem, zorder=3, solid_capstyle="butt")
                sg = 1 if up else -1
                for f in range(flags):
                    fy = ye - sg * 0.9 * f
                    path = p5_Path([(sx, Y(fy, s)), (sx + 0.2, Y(fy - sg * 0.9, s)), (sx + 1.4, Y(fy - sg * 1.3, s)), (sx + 0.95, Y(fy - sg * 2.5, s))],
                                   [p5_Path.MOVETO, p5_Path.CURVE4, p5_Path.CURVE4, p5_Path.CURVE4])
                    ax.add_patch(p5_PathPatch(path, facecolor="none", edgecolor=col, lw=0.13 * pt, zorder=3))
            if (mi, pi, v) in plan:
                accs.append((y, plan[(mi, pi, v)], col))
            if tie_in and (si, v) in prev_head and prev_head[(si, v)][2] == s:
                x1_, y1_, _, up1 = prev_head[(si, v)]
                d = -0.55 if up1 else 0.55
                path = p5_Path([(x1_ + 0.7, Y(y1_ + d, s)), ((x1_ + xh) / 2, Y(y1_ + d + (-1.3 if up1 else 1.3), s)), (xh - 0.7, Y(y + d, s))],
                               [p5_Path.MOVETO, p5_Path.CURVE3, p5_Path.CURVE3])
                ax.add_patch(p5_PathPatch(path, facecolor="none", edgecolor="black", lw=0.1 * pt, zorder=3))
            if tie_out:
                prev_head[(si, v)] = (xh, y, s, up)
        placed = []
        for y, alt, col in sorted(accs, key=lambda a: -a[0]):
            c = 0
            while any(abs(y - y2) < 3.0 and c2 == c for y2, c2 in placed):
                c += 1
            placed.append((y, c))
            glyph = {1: "♯", -1: "♭", 0: "♮"}[alt]
            ax.text(xc - 1.2 - 1.1 * c, Y(y + (0.35 if alt == -1 else 0.0), s), glyph, fontsize=3.4 * pt, color=col,
                    ha="center", va="center", zorder=4)

In [7]:
# P5-06 Quantum report: resource table, violations, energy-per-window chart, circuit diagram
# The report is a plain dict produced by the pipeline (Parts 2-4). Every key is optional - Part 5 shows what exists.
# Keys (aliases in brackets):  solver, backend, is_mock, qubits, depth, two_qubit_gates [gates], shots, layers_p [p],
#   runtime_s, violations (int), violation_details [{rule, slot (0-based), voices (0=bass..3=soprano, optional)}],
#   energy, classical_energy, approx_ratio, p_optimum, feasible_fraction,
#   windows [{start, end (exclusive), qubits, depth, two_qubit_gates, energy, classical_energy}],
#   circuit (qiskit QuantumCircuit or list of them), hardware {dict of a recorded real-device run}, verdict, notes [str]
P5_CARD = ("font-family:system-ui,Segoe UI,Arial,sans-serif;border:2px solid {c};border-radius:10px;padding:14px 18px;"
           "background:#fafafa;color:#222;max-width:920px;margin:6px 0;line-height:1.5")
P5_TD = "padding:3px 14px 3px 0;vertical-align:top"


def p5_pick(d, *keys, default=None):
    for k in keys:
        if isinstance(d, dict) and d.get(k) is not None:
            return d[k]
    return default


def p5_fmt(v, nd=3):
    if v is None:
        return p5_t("rep_na")
    if isinstance(v, bool):
        return "yes" if v else "no"
    if isinstance(v, int) or (isinstance(v, float) and float(v).is_integer() and abs(v) < 1e9):
        return f"{int(v):,}"
    if isinstance(v, float):
        return f"{v:.{nd}g}"
    return p5_html.escape(str(v))


def p5_marks_from_report(report):
    """violation_details -> {slot: set(voices) | None}, used to colour notes red in the score."""
    marks = {}
    for d in (report or {}).get("violation_details") or []:
        try:
            slot = int(d["slot"])
        except (KeyError, TypeError, ValueError):
            continue
        voices = d.get("voices")
        if voices:
            if marks.get(slot, set()) is not None:
                marks.setdefault(slot, set()).update(int(v) for v in voices)
        else:
            marks[slot] = None
    return marks


def p5_windows_from_report(report):
    """windows -> [(start, end_exclusive)] for the shaded bands in the score."""
    out = []
    for w in (report or {}).get("windows") or []:
        if "start" in w and "end" in w:
            out.append((int(w["start"]), int(w["end"])))
        elif w.get("slots"):
            out.append((int(min(w["slots"])), int(max(w["slots"])) + 1))
    return out


def p5_report_html(report, sc=None, pipeline_name="", seconds=None):
    r = report or {}
    is_mock = bool(r.get("is_mock"))
    color = "#c62828" if is_mock else "#2e7d32"
    n_slots = len(sc.slots) if sc is not None else None
    rows = [("rep_pipeline", pipeline_name or None), ("rep_solver", r.get("solver")), ("rep_backend", r.get("backend")),
            ("rep_slots", n_slots), ("rep_qubits", r.get("qubits")), ("rep_depth", r.get("depth")),
            ("rep_2q", p5_pick(r, "two_qubit_gates", "gates", "n_2q_gates")), ("rep_shots", r.get("shots")),
            ("rep_layers", p5_pick(r, "layers_p", "p")), ("rep_runtime", p5_pick(r, "runtime_s", default=seconds)),
            ("rep_viol", r.get("violations")), ("rep_energy", r.get("energy")),
            ("rep_classical", r.get("classical_energy")), ("rep_ratio", r.get("approx_ratio")),
            ("rep_popt", r.get("p_optimum")), ("rep_feas", r.get("feasible_fraction"))]
    h = [f'<div style="{P5_CARD.format(c=color)}">',
         f'<div style="font-size:1.15em;font-weight:700;color:{color};margin-bottom:6px">📊 {p5_html.escape(p5_t("rep_title"))}</div>']
    if is_mock:
        h.append('<div style="background:#ffebee;color:#b71c1c;border-radius:6px;padding:6px 10px;margin:6px 0;font-weight:600">'
                 f'⚠ {p5_html.escape(p5_t("mock_banner"))}</div>')
    h.append('<table style="border-collapse:collapse">')
    for key, val in rows:
        if val is not None:
            h.append(f'<tr><td style="{P5_TD};color:#555">{p5_html.escape(p5_t(key))}</td><td style="{P5_TD};font-weight:600">{p5_fmt(val)}</td></tr>')
    h.append("</table>")
    details = r.get("violation_details") or []
    if details:
        h.append(f'<div style="margin-top:8px"><b>{p5_html.escape(p5_t("rep_viol_head"))}</b><ul style="margin:4px 0 0 18px">')
        for d in details[:12]:
            v = d.get("voices")
            h.append(f'<li>{p5_html.escape(str(d.get("rule", "?")))} - slot {int(d.get("slot", -1)) + 1}'
                     + (f' (voices {", ".join(P5_VOICE_NAMES[int(i)] for i in v)})' if v else "") + "</li>")
        h.append("</ul></div>")
    elif r.get("violations") == 0:
        h.append(f'<div style="margin-top:8px;color:#2e7d32">✔ {p5_html.escape(p5_t("rep_viol_none"))}</div>')
    wins = r.get("windows") or []
    if wins:
        h.append(f'<div style="margin-top:10px"><b>{p5_html.escape(p5_t("rep_windows"))}</b></div>'
                 '<table style="border-collapse:collapse;font-size:.92em"><tr style="color:#555">')
        heads = [p5_t("rep_win_no"), p5_t("rep_win_slots"), p5_t("rep_qubits").split(" (")[0], p5_t("rep_depth"),
                 p5_t("rep_2q"), p5_t("rep_energy"), p5_t("rep_classical"), p5_t("rep_win_gap")]
        h.append("".join(f'<th style="{P5_TD};text-align:left">{p5_html.escape(x)}</th>' for x in heads) + "</tr>")
        for i, w in enumerate(wins, 1):
            e, c = w.get("energy"), w.get("classical_energy")
            gap = (e - c) if isinstance(e, (int, float)) and isinstance(c, (int, float)) else None
            span = f'{int(w["start"]) + 1}-{int(w["end"])}' if "start" in w and "end" in w else p5_t("rep_na")
            cells = [f"W{i}", span, w.get("qubits"), w.get("depth"), p5_pick(w, "two_qubit_gates", "gates"), e, c, gap]
            h.append("<tr>" + "".join(f'<td style="{P5_TD}">{p5_fmt(x) if j else x}</td>' for j, x in enumerate(cells)) + "</tr>")
        h.append("</table>")
    hw = r.get("hardware")
    if isinstance(hw, dict) and hw:
        h.append(f'<div style="margin-top:10px"><b>{p5_html.escape(p5_t("rep_hardware"))}</b><table style="border-collapse:collapse">')
        for k, v in hw.items():
            h.append(f'<tr><td style="{P5_TD};color:#555">{p5_html.escape(str(k))}</td><td style="{P5_TD}">{p5_fmt(v)}</td></tr>')
        h.append("</table></div>")
    if r.get("verdict"):
        h.append(f'<div style="margin-top:10px"><b>{p5_html.escape(p5_t("rep_verdict"))}:</b> {p5_html.escape(str(r["verdict"]))}</div>')
    for note in r.get("notes") or []:
        h.append(f'<div style="margin-top:4px;color:#555">• {p5_html.escape(str(note))}</div>')
    h.append('<hr style="border:0;border-top:1px solid #ddd;margin:12px 0 8px">'
             f'<div style="font-size:.93em;color:#444"><b>💡 {p5_html.escape(p5_t("rep_honest_title"))}</b><br>{p5_html.escape(p5_t("rep_honest"))}</div></div>')
    return "".join(h)


def p5_energy_chart_png(report, width_in=8.0):
    """Grouped bars: quantum energy vs classical optimum for every window (None if there is nothing to plot)."""
    wins = [w for w in (report or {}).get("windows") or [] if isinstance(w.get("energy"), (int, float))]
    if not wins:
        return None
    n = len(wins)
    fig = p5_Figure(figsize=(max(5.0, min(width_in, 1.0 + 0.9 * n)), 3.2), dpi=110, facecolor="white")
    ax = fig.add_subplot(111)
    xs = p5_np.arange(n)
    ax.bar(xs - 0.2, [w["energy"] for w in wins], 0.4, label=p5_t("rep_q_series"), color="#1565c0")
    cl = [w.get("classical_energy") for w in wins]
    if any(isinstance(c, (int, float)) for c in cl):
        ax.bar(xs + 0.2, [c if isinstance(c, (int, float)) else 0 for c in cl], 0.4, label=p5_t("rep_c_series"), color="#9e9e9e")
    ax.set_xticks(xs)
    ax.set_xticklabels([f"W{i + 1}" for i in range(n)])
    ax.set_title(p5_t("rep_chart"), fontsize=10)
    ax.legend(fontsize=8, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    buf = p5_io.BytesIO()
    p5_Canvas(fig).print_png(buf)
    return buf.getvalue()


def p5_circuit_png(circ):
    """PNG of a Qiskit circuit via circ.draw('mpl'); returns None if drawing is not possible."""
    try:
        import matplotlib.pyplot as p5_plt_local
        first = circ[0] if isinstance(circ, (list, tuple)) else circ
        fig = first.draw("mpl", fold=40, scale=0.6)
        buf = p5_io.BytesIO()
        fig.savefig(buf, format="png", dpi=130, bbox_inches="tight", facecolor="white")
        p5_plt_local.close(fig)
        return buf.getvalue()
    except Exception:
        return None


def p5_circuit_text(circ, max_chars=3000):
    try:
        first = circ[0] if isinstance(circ, (list, tuple)) else circ
        txt = str(first.draw("text", fold=120))
        return txt[:max_chars] + ("\n..." if len(txt) > max_chars else "")
    except Exception:
        return None


def p5_show_report(report, sc=None, pipeline_name="", seconds=None):
    """Display the report card, the energy chart and the circuit diagram."""
    p5_display(p5_HTML(p5_report_html(report, sc, pipeline_name, seconds)))
    png = p5_energy_chart_png(report)
    if png:
        p5_display(p5_Image(data=png, format="png"))
    circ = (report or {}).get("circuit")
    if circ is not None:
        p5_display(p5_HTML(f'<div style="font-family:system-ui,Arial,sans-serif;margin-top:8px"><b>{p5_html.escape(p5_t("rep_circuit"))}</b></div>'))
        cpng = p5_circuit_png(circ)
        if cpng:
            p5_display(p5_Image(data=cpng, format="png"))
        else:
            txt = p5_circuit_text(circ)
            if txt:
                p5_display(p5_HTML(f'<pre style="font-size:10px;line-height:1.1;overflow-x:auto;color:#222;background:#fff">{p5_html.escape(txt)}</pre>'))

In [8]:
# P5-07 Pipeline adapter: the ONE place where Part 5 calls the team's code
# Canonical contract (Parts 1-4 provide it):
#     harmonize_pipeline(melody, style="bach_chorale", style_weight=1.0, solver="xy_qaoa", progress=None, **options)
#         -> (result, report)
# melody  : dict from p5_make_melody (title, preset, notes=[(midi, beats)] or None, key, time_sig, pickup, tempo)
# result  : list of (bass, tenor, alto, soprano) MIDI tuples | dict {'voicings': [...], 'chords': [...], ...} | 4-part music21 Stream
# report  : dict, see cell P5-06.
# Parameter names may differ: only the parameters the function actually accepts are passed (see P5_ARG_ALIASES).
P5_ARG_ALIASES = {
    "style": ("style", "style_name"),
    "style_weight": ("style_weight", "weight", "strictness"),
    "solver": ("solver", "solver_type", "method"),
    "progress": ("progress", "progress_cb", "callback", "on_progress"),
    "layers_p": ("layers_p", "p", "reps", "layers"),
    "shots": ("shots", "n_shots"),
    "window_size": ("window_size", "window", "window_slots"),
    "noisy": ("noisy", "noise", "use_noise"),
}


@p5_dataclass
class P5Request:
    melody: dict
    style: str = "bach_chorale"
    style_weight: float = 1.0
    solver: str = "qaoa"
    options: dict = p5_field(default_factory=dict)      # layers_p, shots, window_size, noisy
    tempo: int = 90


@p5_dataclass
class P5Run:
    request: P5Request
    score: object = None                                 # P5Score
    report: dict = p5_field(default_factory=dict)
    warnings: list = p5_field(default_factory=list)
    error: str = ""
    traceback: str = ""
    seconds: float = 0.0
    pipeline_name: str = ""
    artifacts: dict = p5_field(default_factory=dict)      # rendered PNG / WAV, filled by P5-08
    alternatives: list = p5_field(default_factory=list)   # ranked sampled harmony variants


def p5_find_pipeline():
    """(name, function): the team's harmonize_pipeline / harmonize if defined, else the mock."""
    g = globals()
    for name in ("harmonize_pipeline", "harmonize"):
        fn = g.get(name)
        if callable(fn):
            return name, fn
    fn = g.get("p5_mock_pipeline")
    if callable(fn):
        return "p5_mock_pipeline", fn
    raise RuntimeError("No pipeline found: define harmonize_pipeline(melody, ...) or keep cell P5-10 (mock).")


def p5_invoke_pipeline(fn, req, progress=None):
    values = {"style": req.style, "style_weight": req.style_weight, "solver": req.solver, "progress": progress, **req.options}
    params = p5_inspect.signature(fn).parameters
    first = next(iter(params), None)
    has_var_kw = any(p.kind == p.VAR_KEYWORD for p in params.values())
    ok_kinds = (p5_inspect.Parameter.POSITIONAL_OR_KEYWORD, p5_inspect.Parameter.KEYWORD_ONLY)
    kwargs = {}
    for canon, aliases in P5_ARG_ALIASES.items():
        val = values.get(canon)
        if val is None:
            continue
        for al in aliases:
            if al in params and al != first and params[al].kind in ok_kinds and al not in kwargs:
                kwargs[al] = val
                break
        else:
            if has_var_kw:
                kwargs[canon] = val
    return fn(req.melody, **kwargs)


def p5_split_return(raw):
    """Accept (result, report) | {'result':..., 'report':...} | result."""
    if isinstance(raw, tuple) and len(raw) == 2 and isinstance(raw[1], dict) and "pitches" not in raw[1]:
        return raw[0], raw[1]
    if isinstance(raw, dict) and "report" in raw:
        rep = raw["report"]
        res = raw.get("result", {k: v for k, v in raw.items() if k != "report"})
        return res, rep
    return raw, {}


def p5_run_request(req, progress=None):
    """Call the pipeline and normalize its answer. Never raises: errors are stored in the returned P5Run."""
    run = P5Run(request=req)
    t0 = p5_time.time()
    try:
        if req.options.get("use_api"):
            name = "person4 FastAPI QAOA service"
            run.pipeline_name = name
            raw = p5_api_harmonize_pipeline(req, progress)
        else:
            name, fn = p5_find_pipeline()
            run.pipeline_name = name
            solver_melody = dict(req.melody)
            rest_flags = [m is None for m, _ in (req.melody.get("notes") or [])]
            if any(rest_flags):
                carried, previous = [], None
                for midi, duration in req.melody["notes"]:
                    if midi is not None:
                        previous = int(midi)
                    if previous is None:
                        raise ValueError("A melody cannot begin with a rest.")
                    carried.append((previous, duration))
                solver_melody["notes"] = carried
            solver_req = P5Request(melody=solver_melody, style=req.style, style_weight=req.style_weight,
                                   solver=req.solver, options=req.options, tempo=req.tempo)
            raw = p5_invoke_pipeline(fn, solver_req, progress)
        result, report = p5_split_return(raw)
        score_melody = req.melody
        if req.options.get("use_api") and isinstance(result, dict) and result.get("_p5_input_notes") is not None:
            score_melody = dict(req.melody)
            score_melody["notes"] = result["_p5_input_notes"]
        elif any(m is None for m, _ in (req.melody.get("notes") or [])):
            score_melody = dict(req.melody)
            score_melody["notes"] = solver_melody["notes"]
        sc = p5_normalize_result(result, score_melody)
        if req.options.get("use_api") and isinstance(result, dict):
            rest_flags = result.get("_p5_soprano_rests") or []
            if len(rest_flags) != len(sc.slots):
                raise RuntimeError("API soprano rest map does not match the returned chord slots.")
        if len(rest_flags) == len(sc.slots):
            for slot, is_rest in zip(sc.slots, rest_flags):
                slot.soprano_rest = bool(is_rest)
        if req.options.get("use_api") and isinstance(result, dict):
            for alternative in result.get("_p5_alternatives") or []:
                alt_melody = dict(req.melody)
                alt_melody["notes"] = alternative["_p5_input_notes"]
                alt_score = p5_normalize_result(alternative, alt_melody)
                alt_rests = alternative.get("_p5_soprano_rests") or []
                if len(alt_rests) != len(alt_score.slots):
                    raise RuntimeError("Alternative soprano rest map does not match the returned chord slots.")
                for slot, is_rest in zip(alt_score.slots, alt_rests):
                    slot.soprano_rest = bool(is_rest)
                alt_score.tempo = int(req.tempo or alt_score.tempo)
                alt_report = dict(report or {})
                alt_report["energy"] = alternative.get("energy")
                alt_report["quality_report"] = alternative.get("quality_report") or {}
                alt_report["rank"] = alternative.get("rank")
                run.alternatives.append(P5Run(request=req, score=alt_score, report=alt_report,
                                              warnings=list(alt_score.warnings), seconds=run.seconds,
                                              pipeline_name=name))
        sc.tempo = int(req.tempo or sc.tempo)
        run.score, run.report, run.warnings = sc, dict(report or {}), list(sc.warnings)
        if name == "p5_mock_pipeline":
            run.report["is_mock"] = True
    except Exception as e:
        run.error = f"{type(e).__name__}: {e}"
        run.traceback = p5_tb.format_exc()
    run.seconds = p5_time.time() - t0
    return run

In [9]:
# P5-08 Display panels: score + audio, export. Used by the interactive UI and by the static showcase.
def p5_banner_html(text_html, kind="warn"):
    bg, fg = {"warn": ("#fff8e1", "#8d6e00"), "err": ("#ffebee", "#b71c1c"), "mock": ("#ffebee", "#b71c1c"),
              "ok": ("#e8f5e9", "#1b5e20")}[kind]
    return (f'<div style="font-family:system-ui,Segoe UI,Arial,sans-serif;background:{bg};color:{fg};border-radius:6px;'
            f'padding:8px 12px;margin:6px 0;max-width:920px;line-height:1.45">{text_html}</div>')


def p5_render_artifacts(run):
    """Render (once) the score PNG and the WAV files of a successful run."""
    if run.artifacts:
        return run.artifacts
    sc = run.score
    marks = p5_marks_from_report(run.report)
    wins = p5_windows_from_report(run.report)
    title = f"{sc.title or p5_t('score_title')} · {sc.key} major · {sc.time_sig[0]}/{sc.time_sig[1]}"
    run.artifacts = {
        "marks": marks, "windows": wins,
        "png": p5_engrave_png(sc, marks=marks, windows=wins, title=title),
        "wav_full": p5_synth_wav(sc),
        "wav_sop": p5_synth_wav(sc, voices=(3,)),
    }
    return run.artifacts


def p5_show_error(run):
    msg = p5_html.escape(run.error)
    p5_display(p5_HTML(p5_banner_html(f"<b>{p5_html.escape(p5_t('err_title'))}</b><br>{msg}", "err")
                       + f'<details style="max-width:920px"><summary>traceback</summary><pre style="font-size:11px;overflow-x:auto;'
                         f'color:#222;background:#fff">{p5_html.escape(run.traceback)}</pre></details>'))


def p5_show_score_panel(run):
    """Banners, score image and audio players."""
    if run.error:
        p5_show_error(run)
        return
    if run.report.get("is_mock"):
        p5_display(p5_HTML(p5_banner_html("⚠ " + p5_html.escape(p5_t("mock_banner")), "mock")))
    if run.warnings:
        items = "".join(f"<li>{p5_html.escape(w)}</li>" for w in run.warnings)
        p5_display(p5_HTML(p5_banner_html(f"<b>{p5_html.escape(p5_t('warn_title'))}</b><ul style='margin:4px 0 0 18px'>{items}</ul>", "warn")))
    art = p5_render_artifacts(run)
    p5_display(p5_Image(data=art["png"], format="png"))
    if art["marks"]:
        p5_display(p5_HTML(p5_banner_html(p5_html.escape(p5_t("viol_marked")), "warn")))
    p5_display(p5_HTML(p5_audio_html(art["wav_sop"], p5_t("audio_sop")) + p5_audio_html(art["wav_full"], p5_t("audio_full"))))


def p5_show_report_panel(run):
    if run.error:
        p5_show_error(run)
        return
    p5_show_report(run.report, run.score, run.pipeline_name, run.seconds)


def p5_show_export_panel(run):
    """Save MIDI / MusicXML / WAV / PNG to P5_OUT_DIR and show download buttons."""
    if run.error:
        return
    art = p5_render_artifacts(run)
    if p5_m21 is None:
        p5_display(p5_HTML(p5_banner_html(p5_html.escape(p5_t("no_m21")), "warn")))
    files, errors = p5_save_all(run.score, marks=art["marks"], score_png=art["png"], wav=art["wav_full"])
    run.artifacts["files"] = {k: v[0] for k, v in files.items()}
    parts = [p5_download_links_html(files)]
    if files:
        folder = p5_html.escape(p5_os.path.abspath(p5_os.path.dirname(next(iter(files.values()))[0])))
        names = "".join(f"<li><code>{p5_html.escape(p5_os.path.basename(v[0]))}</code></li>" for v in files.values())
        parts.append(f'<div style="font-family:system-ui,Arial,sans-serif;font-size:13px;color:#222;margin-top:6px">'
                     f'{p5_html.escape(p5_t("export_saved"))} <code>{folder}</code><ul style="margin:4px 0 0 18px">{names}</ul></div>')
    for e in errors:
        parts.append(p5_banner_html(p5_html.escape(e), "warn"))
    p5_display(p5_HTML("".join(parts)))


def p5_panel_error_html(run):
    return p5_banner_html(f"<b>{p5_html.escape(p5_t('err_title'))}</b><br>"
                          f"{p5_html.escape(run.error)}<details><summary>traceback</summary><pre>"
                          f"{p5_html.escape(run.traceback)}</pre></details>", "err")


def p5_score_panel_html(run):
    """Build score content as one HTML value, reliable inside widget callbacks."""
    if run.error:
        return p5_panel_error_html(run)
    parts = []
    if run.report.get("is_mock"):
        parts.append(p5_banner_html("âš  " + p5_html.escape(p5_t("mock_banner")), "mock"))
    if run.warnings:
        items = "".join(f"<li>{p5_html.escape(w)}</li>" for w in run.warnings)
        parts.append(p5_banner_html(f"<b>{p5_html.escape(p5_t('warn_title'))}</b><ul>{items}</ul>", "warn"))
    art = p5_render_artifacts(run)
    parts.append(f'<img style="max-width:100%;height:auto" alt="Four-part score" src="data:image/png;base64,'
                 f'{p5_b64.b64encode(art["png"]).decode("ascii")}">')
    if art["marks"]:
        parts.append(p5_banner_html(p5_html.escape(p5_t("viol_marked")), "warn"))
    parts.append(p5_audio_html(art["wav_sop"], p5_t("audio_sop")))
    parts.append(p5_audio_html(art["wav_full"], p5_t("audio_full")))
    return "".join(parts)


def p5_report_panel_html(run):
    if run.error:
        return p5_panel_error_html(run)
    parts = [p5_report_html(run.report, run.score, run.pipeline_name, run.seconds)]
    chart = p5_energy_chart_png(run.report)
    if chart:
        parts.append(f'<img style="max-width:100%;height:auto" alt="Energy per window" src="data:image/png;base64,'
                     f'{p5_b64.b64encode(chart).decode("ascii")}">')
    circ = run.report.get("circuit")
    if circ is not None:
        parts.append(f'<div><b>{p5_html.escape(p5_t("rep_circuit"))}</b></div>')
        cpng = p5_circuit_png(circ)
        if cpng:
            parts.append(f'<img style="max-width:100%;height:auto" alt="QAOA circuit" src="data:image/png;base64,'
                         f'{p5_b64.b64encode(cpng).decode("ascii")}">')
        else:
            txt = p5_circuit_text(circ)
            if txt:
                parts.append(f'<pre style="overflow-x:auto">{p5_html.escape(txt)}</pre>')
    return "".join(parts)


def p5_export_panel_html(run):
    if run.error:
        return p5_panel_error_html(run)
    art = p5_render_artifacts(run)
    parts = []
    if p5_m21 is None:
        parts.append(p5_banner_html(p5_html.escape(p5_t("no_m21")), "warn"))
    files, errors = p5_save_all(run.score, marks=art["marks"], score_png=art["png"], wav=art["wav_full"])
    run.artifacts["files"] = {k: v[0] for k, v in files.items()}
    parts.append(p5_download_links_html(files))
    if files:
        folder = p5_html.escape(p5_os.path.abspath(p5_os.path.dirname(next(iter(files.values()))[0])))
        names = "".join(f"<li><code>{p5_html.escape(p5_os.path.basename(v[0]))}</code></li>" for v in files.values())
        parts.append(f'<div>{p5_html.escape(p5_t("export_saved"))} <code>{folder}</code><ul>{names}</ul></div>')
    parts.extend(p5_banner_html(p5_html.escape(e), "warn") for e in errors)
    return "".join(parts)


def p5_show_run(run):
    """Show everything for one run, one after another (used by the static showcase)."""
    p5_show_score_panel(run)
    if not run.error:
        p5_show_report_panel(run)
        p5_show_export_panel(run)

In [10]:
# P5-09 Interactive panel (ipywidgets): p5_launch() builds and shows it
P5_UI = {}                                   # widgets of the last panel, by name (handy for debugging)
P5_STATE = {"last_run": None, "status": ""}  # P5_STATE['last_run'] is the most recent P5Run


def p5_ui_read_upload(w):
    """FileUpload value -> (name, bytes) for ipywidgets 7 (dict) and 8 (tuple of dicts); None if empty."""
    v = w.value
    if not v:
        return None
    if isinstance(v, dict):
        name = next(iter(v))
        return name, bytes(v[name]["content"])
    item = v[0]
    return item["name"], bytes(item["content"])


def p5_ui_collect(ui):
    """Read the widgets and build a P5Request. Raises ValueError with a readable message on bad input."""
    src = ui["melody"].value
    tempo = int(ui["tempo"].value)
    if src in P5_PRESETS:
        melody = p5_preset_melody(src)
        melody["tempo"] = tempo
    else:
        ts = tuple(int(x) for x in ui["ts"].value.split("/"))
        key = None if ui["key"].value == "auto" else ui["key"].value
        if src == "text":
            notes, errs = p5_parse_melody_text(ui["text"].value)
            if errs:
                raise ValueError("\n".join(errs[:6]))
            melody = p5_make_melody(notes, key=key, time_sig=ts, pickup=ui["pickup"].value, title="Custom melody",
                                    tempo=tempo, source="text")
        else:
            up = p5_ui_read_upload(ui["upload"])
            if up is None:
                raise ValueError("Choose a .mid file first.")
            melody = p5_parse_midi_bytes(up[1], title=p5_os.path.splitext(up[0])[0])
            if key:
                melody["key"] = key
            melody["tempo"] = tempo
        errs, _ = p5_validate_melody(melody["notes"], melody["time_sig"])
        if errs:
            raise ValueError("\n".join(errs[:6]))
    options = {"layers_p": int(ui["p"].value), "shots": int(ui["shots"].value),
               "window_size": int(ui["window"].value), "noisy": ui["backend"].value == "noisy",
               "use_api": bool(ui["use_api"].value), "backend": ui["backend"].value}
    return P5Request(melody=melody, style=ui["style"].value, style_weight=1.0,
                     solver="qaoa", options=options, tempo=tempo)


def p5_safe_panel(out, fn, run):
    """Assign a complete HTML value; avoids Output capture races in callbacks."""
    try:
        out.value = fn(run)
    except Exception:
        out.value = p5_banner_html("<b>Display error</b><pre style='font-size:11px;overflow-x:auto'>"
                                   + p5_html.escape(p5_tb.format_exc()) + "</pre>", "err")


def p5_build_ui():
    """Create all widgets and wire the callbacks. Returns the root widget (also stored in P5_UI['root'])."""
    W = p5_w
    lay = lambda w="520px": W.Layout(width=w)
    sty = {"description_width": "130px"}
    ui = {}
    ui["header"] = W.HTML(f'<div style="font-family:system-ui,Arial,sans-serif"><h3 style="margin:0 0 4px 0">🎵 {p5_html.escape(p5_t("title"))}</h3>'
                          f'<div style="color:#666;font-size:13px">{p5_html.escape(p5_t("subtitle"))}</div></div>')
    options = [(p5_t(spec["label_key"]), pid) for pid, spec in P5_PRESETS.items()]
    options += [(p5_t("opt_custom"), "text"), (p5_t("opt_midi"), "midi")]
    ui["melody"] = W.Dropdown(options=options, value="ode_to_joy", description=p5_t("melody"), layout=lay("470px"), style=sty)

    ui["text"] = W.Textarea(value="E4 E4 F4 G4 G4 F4 E4 D4 C4 C4 D4 E4 E4:1.5 D4:0.5 D4:2", layout=W.Layout(width="620px", height="64px"),
                            continuous_update=False)
    ui["hint"] = W.HTML(f'<div style="color:#666;font-size:12px">{p5_html.escape(p5_t("custom_hint"))}</div>')
    ui["pal_dur"] = W.Dropdown(options=[("1", 1.0), ("½", 0.5), ("¼", 0.25), ("1½", 1.5), ("2", 2.0), ("3", 3.0), ("4", 4.0)], value=1.0,
                               description=p5_t("pal_dur"), layout=lay("230px"), style={"description_width": "110px"})
    ui["pal_acc"] = W.Dropdown(options=[("♮", ""), ("♯", "#"), ("♭", "b")], value="", description=p5_t("pal_acc"),
                               layout=lay("200px"), style={"description_width": "90px"})
    ui["undo"] = W.Button(description=p5_t("btn_undo"), layout=lay("80px"))
    ui["clear"] = W.Button(description=p5_t("btn_clear"), layout=lay("80px"))
    keys = [f"{n}{o}" for o in (4, 5) for n in "CDEFGAB"] + ["C6"]
    ui["keys"] = [W.Button(description=k, layout=W.Layout(width="46px", height="30px", padding="0")) for k in keys]
    ui["palette"] = W.HBox(ui["keys"], layout=W.Layout(flex_flow="row wrap", width="760px"))
    ui["text_box"] = W.VBox([ui["text"], ui["hint"], W.HBox([ui["pal_dur"], ui["pal_acc"], ui["undo"], ui["clear"]]), ui["palette"]])

    ui["upload"] = W.FileUpload(accept=".mid,.midi", multiple=False, description=p5_t("opt_midi"), layout=lay("220px"))
    ui["midi_box"] = W.VBox([ui["upload"]])
    ui["key"] = W.Dropdown(options=[(p5_t("key_auto"), "auto")] + [(k.replace("b", "♭"), k) for k in P5_KEY_FIFTHS],
                           value="auto", description=p5_t("key_label"), layout=lay("260px"), style={"description_width": "110px"})
    ui["ts"] = W.Dropdown(options=["4/4", "3/4", "2/4", "6/8"], value="4/4", description=p5_t("ts_label"), layout=lay("230px"),
                          style={"description_width": "100px"})
    ui["pickup"] = W.BoundedFloatText(value=0.0, min=0.0, max=7.0, step=0.5, description=p5_t("pickup_label"), layout=lay("210px"),
                                      style={"description_width": "100px"})
    ui["custom_opts"] = W.HBox([ui["key"], ui["ts"], ui["pickup"]])

    ui["style"] = W.Dropdown(options=[("Bach chorale", "bach_chorale")], value="bach_chorale", disabled=True,
                             description=p5_t("style_label"), layout=lay("320px"), style=sty)
    ui["solver"] = W.HTML(f'<div style="font-family:system-ui,Arial,sans-serif;margin:4px 0 8px 0">'
                          f'<span style="display:inline-block;width:130px">{p5_html.escape(p5_t("solver_label"))}</span>'
                          f'<b>{p5_html.escape(p5_t("s_qaoa"))}</b></div>')
    ui["tempo"] = W.IntSlider(value=100, min=50, max=160, step=2, description=p5_t("tempo_label"), layout=lay("470px"), style=sty)
    ui["p"] = W.IntSlider(value=2, min=1, max=5, description=p5_t("adv_p"), layout=lay("420px"), style=sty)
    ui["shots"] = W.Dropdown(options=[512, 1024, 2048, 4096, 8192], value=2048, description=p5_t("adv_shots"), layout=lay("300px"), style=sty)
    ui["window"] = W.IntSlider(value=3, min=2, max=4, description=p5_t("adv_window"), layout=lay("420px"), style=sty)
    ui["use_api"] = W.Checkbox(value=True, description="Use local quantum API", indent=False)
    ui["backend"] = W.Dropdown(options=[("Aer simulator", "aer"), ("Noisy fake backend", "noisy"),
                                        ("IBM Quantum hardware", "ibm")], value="aer",
                                description="Quantum backend", layout=lay("320px"), style=sty)
    ui["adv"] = W.Accordion(children=[W.VBox([ui["use_api"], ui["backend"], ui["p"], ui["shots"], ui["window"]])], selected_index=0)
    ui["adv"].set_title(0, p5_t("adv_title"))

    ui["preview"] = W.Button(description=p5_t("btn_preview"), icon="volume-up", layout=lay("190px"))
    ui["run"] = W.Button(description=p5_t("btn_run"), button_style="success", icon="play", layout=lay("230px"))
    ui["progress"] = W.IntProgress(value=0, min=0, max=100, layout=lay("260px"))
    ui["status"] = W.Label(value=p5_t("status_ready"))
    ui["variant"] = W.Dropdown(options=[(p5_t("variant_best"), 0)], value=0,
                               description=p5_t("variant_label"), disabled=True,
                               layout=lay("420px"), style=sty)
    ui["out_score"], ui["out_report"], ui["out_export"] = W.HTML(), W.HTML(), W.HTML()
    ui["tabs"] = W.Tab(children=[ui["out_score"], ui["out_report"], ui["out_export"]])
    for i, k in enumerate(("tab_score", "tab_report", "tab_export")):
        ui["tabs"].set_title(i, p5_t(k))

    sec = lambda k: W.HTML(f'<div style="font-family:system-ui,Arial,sans-serif;font-weight:600;margin-top:8px">{p5_html.escape(p5_t(k))}</div>')
    ui["root"] = W.VBox([ui["header"], sec("sec_melody"), ui["melody"], ui["text_box"], ui["midi_box"], ui["custom_opts"],
                         sec("sec_style"), ui["style"], ui["solver"], ui["tempo"], ui["adv"],
                         W.HBox([ui["preview"], ui["run"]]), W.HBox([ui["progress"], ui["status"]]), ui["variant"], ui["tabs"]])

    def show_hide(_=None):
        v = ui["melody"].value
        ui["text_box"].layout.display = None if v == "text" else "none"
        ui["midi_box"].layout.display = None if v == "midi" else "none"
        ui["custom_opts"].layout.display = None if v in ("text", "midi") else "none"
        if v in P5_PRESETS:
            ui["tempo"].value = P5_PRESETS[v]["tempo"]
    ui["melody"].observe(show_hide, names="value")
    show_hide()

    def add_note(btn):
        name = btn.description
        tok = name[0] + ui["pal_acc"].value + name[1:]
        dur = float(ui["pal_dur"].value)
        dur_text = {0.25: "s", 0.5: "e", 0.75: "e.", 1.5: "q.", 2.0: "h", 3.0: "h.", 4.0: "w"}.get(dur)
        if dur_text:
            tok += f":{dur_text}"
        elif dur != 1.0:
            tok += f":{dur:g}"
        ui["text"].value = (ui["text"].value.strip() + " " + tok).strip()
    for b in ui["keys"]:
        b.on_click(add_note)
    ui["undo"].on_click(lambda _b: setattr(ui["text"], "value", " ".join(ui["text"].value.split()[:-1])))
    ui["clear"].on_click(lambda _b: setattr(ui["text"], "value", ""))

    def on_preview(_b):
        try:
            req = p5_ui_collect(ui)
            m = req.melody
            if not m["notes"]:
                ui["out_score"].value = p5_banner_html(p5_html.escape(p5_t("no_notes_preset")), "warn")
                ui["tabs"].selected_index = 0
                return
            prev = next(n for n, _ in m["notes"] if n is not None)
            slots = []
            for n, b in m["notes"]:
                rest = n is None
                if not rest:
                    prev = n
                slots.append(P5Slot((prev, prev, prev, prev), b, soprano_rest=rest))
            sc = P5Score(slots=slots, key=m["key"], time_sig=m["time_sig"],
                         tempo=m["tempo"], pickup=m["pickup"], title=m["title"])
            names = " ".join("R" if n is None else p5_note_name(n, m["key"]) for n, _ in m["notes"])
            info = p5_t("parsed", n=len(m["notes"]), key=m["key"], ts=f"{m['time_sig'][0]}/{m['time_sig'][1]}",
                        beats=sum(b for _, b in m["notes"]))
            _, warns = p5_validate_melody(m["notes"], m["time_sig"])
            ui["out_score"].value = (f'<div style="font-family:system-ui,Arial,sans-serif;color:#222"><b>{p5_html.escape(p5_t("preview_title"))}</b> - '
                                     f'{p5_html.escape(info)}<br><code>{p5_html.escape(names)}</code></div>'
                                     + "".join(p5_banner_html(p5_html.escape(w), "warn") for w in warns[:4])
                                     + p5_audio_html(p5_synth_wav(sc, voices=(3,)), p5_t("audio_sop")))
        except Exception as e:
            ui["out_score"].value = p5_banner_html(p5_html.escape(f"{type(e).__name__}: {e}").replace("\n", "<br>"), "err")
        ui["tabs"].selected_index = 0

    def show_variant(index=None):
        best_run = P5_STATE.get("last_run")
        if best_run is None or best_run.error:
            return
        choices = [best_run] + list(best_run.alternatives or [])
        idx = int(ui["variant"].value if index is None else index)
        if not 0 <= idx < len(choices):
            return
        selected = choices[idx]
        p5_safe_panel(ui["out_score"], p5_score_panel_html, selected)
        p5_safe_panel(ui["out_report"], p5_report_panel_html, selected)
        p5_safe_panel(ui["out_export"], p5_export_panel_html, selected)

    def on_variant_change(change):
        if change.get("name") == "value" and P5_STATE.get("last_run") is not None:
            show_variant(change.get("new"))

    ui["variant"].observe(on_variant_change, names="value")

    def on_run(_b):
        ui["run"].disabled = ui["preview"].disabled = True
        ui["progress"].value = 0
        ui["status"].value = p5_t("status_running")
        try:
            try:
                req = p5_ui_collect(ui)
            except Exception as e:
                run = P5Run(request=None, error=f"{type(e).__name__}: {e}")
            else:
                def progress(frac, msg=""):
                    ui["progress"].value = int(100 * max(0.0, min(1.0, float(frac))))
                    if msg:
                        ui["status"].value = str(msg)
                run = p5_run_request(req, progress)
            P5_STATE["last_run"] = run
            variants = [run] + list(run.alternatives or [])
            options = []
            for i, variant in enumerate(variants):
                energy = variant.report.get("energy")
                progression = []
                slots = variant.score.slots if variant.score is not None else []
                for slot in slots:
                    chord = slot.chord
                    chord = str(chord)
                    if chord and (not progression or progression[-1] != chord):
                        progression.append(chord)
                chord_path = " – ".join(progression[:10])
                if len(progression) > 10:
                    chord_path += " – …"
                label = f"Progression {i + 1}: {chord_path}" if chord_path else f"Progression {i + 1}"
                if energy is not None:
                    label += f" (energy {float(energy):.3g})"
                options.append((label, i))
            ui["variant"].options = options or [(p5_t("variant_best"), 0)]
            ui["variant"].value = 0
            ui["variant"].disabled = bool(run.error) or len(variants) < 2
            if run.error:
                p5_safe_panel(ui["out_score"], p5_panel_error_html, run)
            else:
                show_variant(0)
            P5_STATE["status"] = "ok" if not run.error else "error: " + run.error
            ui["status"].value = p5_t("status_done", sec=run.seconds) if not run.error else p5_t("status_error")
            ui["tabs"].selected_index = 0
        finally:
            ui["progress"].value = 100
            ui["run"].disabled = ui["preview"].disabled = False
    ui["preview"].on_click(on_preview)
    ui["run"].on_click(on_run)
    P5_UI.clear()
    P5_UI.update(ui)
    return ui["root"]


def p5_clear_output_widget(out):
    out.value = ""


def p5_launch(lang=None):
    """Build and display the interactive panel. lang='zh' switches the interface to Chinese."""
    global P5_LANG
    if lang:
        P5_LANG = lang
    p5_display(p5_build_ui())          # widgets live in P5_UI; nothing is returned so the cell shows the panel once

In [11]:
# P5-10 MOCK pipeline - DELETE THIS CELL once the team's harmonize_pipeline exists
# A tiny classical dynamic-programming harmonizer so the interface, score, audio, export and report can be
# tested today. It is NOT quantum and the report says so (is_mock=True).
P5_MOCK_CHORDS = {"I": (0, 4, 7), "ii": (2, 5, 9), "IV": (5, 9, 0), "V": (7, 11, 2), "vi": (9, 0, 4)}


def p5_mock_candidates(sop, tonic, k_keep=8, want_tonic=False):
    rel = (sop - tonic) % 12
    out = []
    for name, cp in P5_MOCK_CHORDS.items():
        tones = [(tonic + c) % 12 for c in cp]
        for b in [x for x in range(40, 61) if x % 12 in tones[:2]]:
            for t in [x for x in range(48, 70) if x % 12 in tones and x > b]:
                for a in [x for x in range(55, 75) if x % 12 in tones and t < x < sop]:
                    if sop - a > 12 or a - t > 12:
                        continue
                    pcs = {b % 12, t % 12, a % 12, sop % 12}
                    cost = (0 if tones[0] in pcs and tones[1] in pcs else 6) + (0 if b % 12 == tones[0] else 2)
                    cost += 12 if rel not in cp else 0     # allow chromatic soprano notes in the UI placeholder
                    cost += 2 * [b, t, a, sop].count(tones[1]) % 3
                    cost += 8 if (want_tonic and name != "I") else 0     # start and end on the tonic chord
                    out.append((cost, name, (b, t, a, sop)))
    out.sort()
    return out[:k_keep]


def p5_mock_parallels(p, q):
    """Voice pairs moving in parallel perfect fifths / octaves from voicing p to q."""
    hits = []
    for i in range(4):
        for j in range(i + 1, 4):
            a, b = (p[j] - p[i]) % 12, (q[j] - q[i]) % 12
            moved = p[i] != q[i] and p[j] != q[j]
            same_dir = (q[i] - p[i]) * (q[j] - p[j]) > 0
            if moved and same_dir and a == b and a in (0, 7):
                hits.append((i, j))
    return hits


def p5_mock_pipeline(melody, style_weight=1.0, solver="viterbi", progress=None):
    notes = melody.get("notes") if isinstance(melody, dict) else None
    if not notes:
        raise ValueError(f"The mock backend has no notes for preset '{(melody or {}).get('preset')}'. "
                         "Its data comes from Part 1 - use Ode to Joy or a custom melody until the real pipeline is in.")
    key = melody.get("key") or "C"
    tonic = p5_key_tonic_pc(key)
    sops = [m for m, _ in notes]
    cands = []
    for i, s in enumerate(sops):
        c = p5_mock_candidates(s, tonic, want_tonic=(i in (0, len(sops) - 1)))
        if not c:
            raise ValueError(f"Mock backend: no voicing found for slot {i + 1} ({p5_note_name(s, key)}).")
        cands.append(c)
        if progress:
            progress(0.4 * (i + 1) / len(sops), f"mock: candidates {i + 1}/{len(sops)}")
    best = [[(c[0], None) for c in cands[0]]]                      # DP table: (cost, backpointer)
    for t in range(1, len(sops)):
        row = []
        for cur in cands[t]:
            opts = []
            for j, prev in enumerate(cands[t - 1]):
                p, q = prev[2], cur[2]
                move = 0.5 * abs(q[0] - p[0]) + abs(q[1] - p[1]) + abs(q[2] - p[2])
                par = 20.0 * style_weight * len(p5_mock_parallels(p, q))
                retro = 3.0 if (prev[1] == "V" and cur[1] in ("IV", "ii")) else (-1.5 if (prev[1] == "V" and cur[1] == "I") else 0.0)
                opts.append((best[t - 1][j][0] + cur[0] + move + par + retro, j))
            row.append(min(opts))
        best.append(row)
        if progress:
            progress(0.4 + 0.6 * t / len(sops), f"mock: dynamic programming {t + 1}/{len(sops)}")
    j = min(range(len(best[-1])), key=lambda k: best[-1][k][0])
    total = best[-1][j][0]
    path = [j]
    for t in range(len(sops) - 1, 0, -1):
        j = best[t][j][1]
        path.append(j)
    path.reverse()
    chosen = [cands[t][path[t]] for t in range(len(sops))]
    voicings = [c[2] for c in chosen]
    details = []
    for t in range(1, len(voicings)):
        for i, j2 in p5_mock_parallels(voicings[t - 1], voicings[t]):
            details.append({"rule": "parallel fifths/octaves", "slot": t, "voices": [i, j2]})
    result = {"voicings": voicings, "chords": [c[1] for c in chosen], "durations": [b for _, b in notes],
              "key": key, "time_sig": melody.get("time_sig"), "pickup": melody.get("pickup"), "tempo": melody.get("tempo"),
              "title": melody.get("title")}
    report = {"is_mock": True, "solver": f"mock classical DP (requested: {solver})", "backend": "none (no quantum circuit)",
              "violations": len(details), "violation_details": details, "energy": round(float(total), 2),
              "notes": ["Placeholder harmonizer used while the real pipeline is not defined."]}
    return result, report

In [12]:
# P5-11 Optional demo showcase and post-merge self-check
P5_DEFAULT_OPTIONS = {"layers_p": 2, "shots": 2048, "window_size": 3, "noisy": False}


def p5_demo_showcase(presets=("ode_to_joy", "o_canada"), solver="qaoa", style="bach_chorale", style_weight=1.0, **options):
    """Run the demo melodies through the pipeline and show score, audio, report and export for each one.
    Render both demo melodies in the current notebook session."""
    opts = {**P5_DEFAULT_OPTIONS, **options}
    for pid in presets:
        spec = P5_PRESETS[pid]
        p5_display(p5_HTML(f'<h3 style="font-family:system-ui,Arial,sans-serif;margin:18px 0 4px 0;color:#222">🎼 {p5_html.escape(p5_t(spec["label_key"]))} '
                           f'- solver: <code>{p5_html.escape(solver)}</code></h3>'))
        melody = p5_preset_melody(pid)
        req = P5Request(melody=melody, style=style, style_weight=style_weight, solver=solver, options=opts, tempo=melody["tempo"])
        run = p5_run_request(req)
        P5_STATE["last_run"] = run
        p5_show_run(run)


def p5_selfcheck(run_pipeline=True):
    """Run after merging cells: checks dependencies, the pipeline hook and every Part-5 stage; prints a checklist."""
    results = []

    def step(name, fn):
        try:
            detail = fn()
            results.append((True, name, detail or ""))
        except Exception as e:
            results.append((False, name, f"{type(e).__name__}: {e}"))

    def deps():
        import importlib
        found = []
        for mod in ("music21", "qiskit", "qiskit_optimization", "qiskit_aer", "pylatexenc"):
            try:
                importlib.import_module(mod)
                found.append(mod)
            except Exception:
                pass
        return "available: " + ", ".join(found) + ("" if p5_m21 is not None else "  (music21 missing -> no MIDI/MusicXML export)")

    def hook():
        name, _ = p5_find_pipeline()
        return name + ("  <- MOCK: real harmonize_pipeline not defined yet" if name == "p5_mock_pipeline" else "")

    def parse():
        notes, errs = p5_parse_melody_text("C4 D4 E4:1.5 F4:0.5 G4 | Bb4")
        assert not errs and len(notes) == 6, errs
        return f"{len(notes)} notes, key guess {p5_guess_key(notes)}"

    def stages():
        sc = P5Score(slots=[P5Slot(v, 1.0, "I") for v in ((48, 55, 64, 72), (43, 55, 62, 71), (48, 55, 64, 72))], key="C", title="selfcheck")
        png = p5_engrave_png(sc)
        wav = p5_synth_wav(sc)
        assert png[:8] == b"\x89PNG\r\n\x1a\n" and wav[:4] == b"RIFF"
        extra = ""
        if p5_m21 is not None:
            assert len(p5_m21_bytes(sc, "midi")) > 50 and b"<score-partwise" in p5_m21_bytes(sc, "musicxml")
            extra = ", MIDI + MusicXML"
        return f"score PNG {len(png) // 1024} KB, WAV {len(wav) // 1024} KB{extra}"

    def smoke():
        melody = p5_make_melody([(60, 1), (62, 1), (64, 1), (60, 1)], key="C", title="smoke test", tempo=90)
        run = p5_run_request(P5Request(melody=melody, solver="viterbi", options=dict(P5_DEFAULT_OPTIONS), tempo=90))
        assert not run.error, run.error
        p5_render_artifacts(run)
        note = f"; warnings: {run.warnings}" if run.warnings else ""
        return f"{len(run.score.slots)} slots via {run.pipeline_name} in {run.seconds:.1f}s{note}"

    step("dependencies", deps)
    step("pipeline hook", hook)
    step("melody parsing", parse)
    step("score / audio / export", stages)
    if run_pipeline:
        step("end-to-end smoke test (4 notes)", smoke)
    for ok, name, detail in results:
        print(("✔ " if ok else "✘ ") + name + (f" - {detail}" if detail else ""))
    return all(ok for ok, _, _ in results)

In [13]:
# P5-15 Local FastAPI bridge: the interactive UI can submit real team-pipeline jobs.
import json as p5_json
import urllib.error as p5_urllib_error
import urllib.request as p5_urllib_request

P5_API_BASE_URL = p5_os.environ.get("P5_API_URL", "http://127.0.0.1:8000").rstrip("/")


def p5_api_json(method, path, payload=None, timeout=20):
    """Small stdlib HTTP client so the notebook does not need another dependency."""
    body = None if payload is None else p5_json.dumps(payload).encode("utf-8")
    request = p5_urllib_request.Request(P5_API_BASE_URL + path, data=body, method=method,
                                        headers={"Content-Type": "application/json"})
    try:
        with p5_urllib_request.urlopen(request, timeout=timeout) as response:
            return p5_json.loads(response.read().decode("utf-8"))
    except p5_urllib_error.HTTPError as exc:
        try:
            detail = p5_json.loads(exc.read().decode("utf-8"))
        except Exception:
            detail = str(exc.reason)
        if isinstance(detail, dict):
            detail = detail.get("detail", detail)
        raise RuntimeError(f"Quantum API returned HTTP {exc.code}: {detail}") from None
    except (p5_urllib_error.URLError, TimeoutError) as exc:
        raise RuntimeError(f"Cannot reach {P5_API_BASE_URL}. Start the local API in a separate terminal. ({exc})") from None


def p5_api_harmonize_pipeline(req, progress=None):
    """Submit one melody to the async API, poll it, and adapt its response to Part 5."""
    notes = req.melody.get("notes")
    if not notes:
        raise ValueError("This melody preset has no note list supported by the API yet. Choose Ode to Joy or enter a melody.")
    if req.solver not in ("qaoa", "xy_qaoa"):
        raise ValueError("The API supports QAOA solvers only. Select QAOA or XY-QAOA.")
    backend = req.options.get("backend", "aer")
    style = {"bach_chorale": "chorale", "strict_counterpoint": "strict", "jazz_loose": "jazz_loose"}.get(req.style)
    if style is None:
        raise ValueError(f"Unsupported style for the API: {req.style}")
    rest_flags = [midi is None for midi, _ in notes]
    carried = []
    previous = next((int(midi) for midi, _ in notes if midi is not None), None)
    for midi, _duration in notes:
        if midi is not None:
            previous = int(midi)
        if previous is None:
            raise ValueError("A melody cannot begin with a rest.")
        carried.append(previous)
    melody = {
        "title": req.melody.get("title") or "Untitled",
        "tonic": int(p5_key_tonic_pc(req.melody.get("key") or "C")),
        "mode": req.melody.get("mode", "major"),
        "time_signature": list(req.melody.get("time_sig") or (4, 4)),
        # Keep each melody event intact in the returned score (for example, a dotted quarter).
        # Melody-note boundaries still create chord slots; the four-beat pulse avoids splitting
        # an event at an unrelated quarter-beat boundary.
        "harmonic_rhythm": 4.0,
        "notes": [{"midi": midi, "duration": float(duration)} for midi, duration in zip(carried, (d for _, d in notes))],
    }
    payload = {"melody": melody, "tempo": int(req.tempo), "style": style,
               "run": {"backend": backend, "solver": req.solver, "shots": int(req.options.get("shots", 2048)),
                       "reps": int(req.options.get("layers_p", 2)), "window_size": int(req.options.get("window_size", 3)),
                       "max_qubits": 16, "max_iterations": 50, "seed": 42},
               # A hardware selection should always submit a fresh provider job.
               "use_cache": backend != "ibm"}
    job = p5_api_json("POST", "/jobs", payload, timeout=30)
    job_id = job.get("job_id")
    if not job_id:
        raise RuntimeError("API accepted the request but returned no job ID.")
    started = p5_time.monotonic()
    while p5_time.monotonic() - started < 315:
        state = p5_api_json("GET", f"/jobs/{job_id}", timeout=20)
        status = state.get("status")
        detail = state.get("progress") or {}
        done, total = detail.get("completed_windows"), detail.get("total_windows")
        frac = (float(done) / float(total)) if isinstance(done, (int, float)) and total else (0.05 if status == "queued" else 0.1)
        if progress:
            progress(min(0.98, frac), detail.get("stage") or f"Quantum API job {status}")
        if status == "failed":
            error = state.get("error") or {}
            raise RuntimeError(error.get("message") or "Quantum API job failed.")
        if status == "cancelled":
            raise RuntimeError("Quantum API job was cancelled.")
        if status == "completed":
            envelope = p5_api_json("GET", f"/jobs/{job_id}/result", timeout=30)
            result = envelope.get("result") or {}
            slots = result.get("slots") or []
            if not slots:
                raise RuntimeError("API job completed but returned no score slots.")
            if progress:
                progress(1.0, "Quantum run complete")
            return p5_api_adapt_result(result, envelope.get("provenance") or {}, req, job_id)
        p5_time.sleep(0.5)
    raise RuntimeError(f"Timed out waiting for API job {job_id}; check {P5_API_BASE_URL}/jobs/{job_id}.")


def p5_api_expand_slots(slots, input_notes):
    """Expand API slots against the melody while preserving soprano rests."""
    voicings = [slot["pitches"] for slot in slots]
    durations = [slot["duration"] for slot in slots]
    chords = [slot.get("chord", "") for slot in slots]
    melody_slots, rest_slots = [], []
    note_index, remaining = 0, (float(input_notes[0][1]) if input_notes else 0.0)
    for slot in slots:
        while note_index < len(input_notes) and remaining <= 1e-7:
            note_index += 1
            if note_index < len(input_notes):
                remaining = float(input_notes[note_index][1])
        if note_index >= len(input_notes):
            raise RuntimeError("API returned more chord duration than the input melody contains.")
        duration = float(slot["duration"])
        source_soprano = input_notes[note_index][0]
        is_rest = source_soprano is None
        soprano = (next((int(m) for m, _ in reversed(input_notes[:note_index]) if m is not None), None)
                   if is_rest else int(source_soprano))
        if soprano is None:
            raise RuntimeError("Melody cannot begin with a rest.")
        if int(slot["pitches"][3]) != soprano:
            raise RuntimeError(f"API changed the soprano melody at chord slot {len(melody_slots) + 1}.")
        if duration > remaining + 1e-7:
            raise RuntimeError("API returned a chord slot that crosses a melody note boundary.")
        melody_slots.append((soprano, duration))
        rest_slots.append(is_rest)
        remaining -= duration
    while note_index < len(input_notes) and remaining <= 1e-7:
        note_index += 1
        if note_index < len(input_notes):
            remaining = float(input_notes[note_index][1])
    if note_index != len(input_notes):
        raise RuntimeError("API chord slots do not cover the complete input melody.")
    return voicings, durations, chords, melody_slots, rest_slots


def p5_api_adapt_result(result, provenance, req, job_id):
    slots = result.get("slots") or []
    input_notes = req.melody.get("notes") or []
    voicings, durations, chords, melody_slots, rest_slots = p5_api_expand_slots(slots, input_notes)
    alternatives = []
    for alternative in (result.get("alternatives") or [])[:2]:
        alt_slots = alternative.get("slots") or []
        if not alt_slots:
            continue
        av, ad, ac, am, ar = p5_api_expand_slots(alt_slots, input_notes)
        alternatives.append({"voicings": av, "durations": ad, "chords": ac,
                            "_p5_input_notes": am, "_p5_soprano_rests": ar,
                            "energy": alternative.get("energy"),
                            "quality_report": alternative.get("quality_report") or {},
                            "window_energies": alternative.get("window_energies") or [],
                            "rank": alternative.get("rank")})
    execution = result.get("execution") or {}
    qreport = result.get("quantum_report") or {}
    quality = result.get("quality_report") or {}
    raw_windows = qreport.get("windows") or []
    windows = []
    resources = []
    for item in raw_windows:
        resource = item.get("resources") or {}
        if resource:
            resources.append(resource)
        windows.append({"start": item.get("start"), "end": item.get("stop"),
                        "qubits": resource.get("n_qubits"), "depth": resource.get("depth"),
                        "two_qubit_gates": resource.get("two_qubit_gates"), "energy": item.get("energy")})
    counts = quality.get("rule_violation_counts")
    violations = sum(counts.values()) if isinstance(counts, dict) else None
    report = {
        "solver": req.solver, "backend": execution.get("mode", req.options.get("backend", "aer")),
        "is_mock": bool(execution.get("is_mock", False)),
        "qubits": qreport.get("max_qubits_per_window"),
        "depth": max((r.get("depth", 0) for r in resources), default=None),
        "two_qubit_gates": max((r.get("two_qubit_gates", 0) for r in resources), default=None),
        "shots": req.options.get("shots"), "layers_p": req.options.get("layers_p"),
        "runtime_s": execution.get("elapsed_seconds"), "violations": violations,
        "energy": quality.get("energy"), "classical_energy": quality.get("optimal_energy"),
        "approx_ratio": None, "p_optimum": None,
        "feasible_fraction": min((w.get("feasible_fraction", 1.0) for w in raw_windows), default=None),
        "windows": windows, "verdict": quality.get("verdict"), "hardware": None,
        "notes": list(result.get("warnings") or []),
    }
    report["notes"].append("Top alternatives are ranked from sampled window solutions; window stitching does not prove a global top-three ranking.")
    report["top_solutions"] = [{"rank": 1, "energy": quality.get("energy")}] + [
        {"rank": item.get("rank"), "energy": item.get("energy"),
         "quality_report": item.get("quality_report"),
         "window_energies": item.get("window_energies")} for item in alternatives]
    report["notes"].append("Circuit diagrams are not included in the API response; reported resources are from transpiled windows.")
    if execution.get("mode") == "ibm":
        report["notes"].append(f"Local API job ID: {job_id}; IBM provider job ID is not exposed by this API.")
    if violations is None:
        report["notes"].append("No independent musical rule checker is configured; rule violations are unknown.")
    if provenance.get("cache_hit"):
        report["notes"].append("Returned from the API result cache.")
    return ({"voicings": voicings, "durations": durations, "chords": chords,
             "key": req.melody.get("key", "C"), "time_sig": req.melody.get("time_sig", (4, 4)),
             "tempo": req.tempo, "title": req.melody.get("title", ""),
             "_p5_input_notes": melody_slots, "_p5_soprano_rests": rest_slots,
             "_p5_alternatives": alternatives}, report)

### Self-check — run once after merging the cells

Prints a ✔/✘ checklist: dependencies, pipeline hook, melody parsing, score/audio/export, and a 4-note end-to-end run.

In [14]:
# P5-12 Self-check
p5_selfcheck()

✔ dependencies - available: music21, qiskit, qiskit_aer, pylatexenc
✔ pipeline hook - p5_mock_pipeline  <- MOCK: real harmonize_pipeline not defined yet
✔ melody parsing - 6 notes, key guess F
✔ score / audio / export - score PNG 19 KB, WAV 79 KB, MIDI + MusicXML
✔ end-to-end smoke test (4 notes) - 4 slots via p5_mock_pipeline in 0.0s


True

### Interactive panel

Pick a melody (or type / click / upload one), then press **Run harmonization**. The panel uses standard QAOA.
Tabs: *Score & audio* · *Quantum report* · *Export* (MIDI, MusicXML, WAV, PNG). Use `p5_launch("zh")` for Chinese.

In [15]:
# P5-13 Launch the panel
p5_launch()